# hexaco-lsh results: the six HEXACO factors as with/without persona traits

Interprets and graphs what `hexaco-lsh.ipynb` left in `~/hexaco-lsh-output` (and its telemetry in
`~/gpu-telemetry/hexaco-lsh_*`). It is `persona-lsh-results.ipynb` with the traits swapped, less
what that run doesn't have, plus what six personality factors call for:

- **Gone:** the judges against gpt-5.4 (nothing published on these rubrics, so the two local judges are checked
  against each other only: section 9), and the comparison with the published design.
- **Added:** which questions and facets fail to elicit (2b: the trait data are candidates, `../PLAN.md` section
  9), the same models on persona-lsh's seven traits (4b), confusions pooled by factor pair (5b), and whether
  the six directions are six or mostly one axis (7a), which is what the self-report directions of
  `persona-hexaco` (dropped 2026-11-24; `../PLAN.md` section 5) turned out to be.

**The run.** Experiment `rand40-t256-s68853742`, captured 2026-11-24 on cards 0-2 in fp16: five models
(Llama-3.1-8B, Granite-4.2-8B, Bolt-E4B, Nemotron-Nano-8B, Qwen3-8B with thinking off), six factors, 20 test and
20 eval questions per factor with a random instruction pair each, 256-token replies, gemma-qat as the gate and
gpt-oss as the second opinion. *With* is the factor's high pole and *without* its low pole: both replies are
instructed, and neither is the model's unprompted answer.

**Run order.** After `hexaco-lsh.ipynb` has finished (its sections 5-7 write `accuracy.csv`, `kept_traits.json`,
`predictions.json`, `layer_sweep.json` and `pairs.json`), run this notebook on the rig with the `gpu` kernel.
Nothing here loads a model or touches a card. `EXPERIMENT = "latest"` opens the newest experiment with an
`experiment.json`.

**Offline, on another machine:** copy `~/hexaco-lsh-output` and `~/gpu-telemetry/hexaco-lsh_*` from the rig to
the same paths (or set `HEXACO_LSH_OUTPUT` and `GPU_TELEMETRY`); `notes.md` has the rsync
commands for persona-lsh, and these are the same with the directory changed. Section 7 and 9b need the `.pt`
vectors and torch (CPU is enough); every other section runs from the JSON and CSV files alone.

| Section | Question | Reads |
|---|---|---|
| 1 | What did each run cost, and did replies fit in their token budget? | `run_info.json`, `responses.json` |
| 2 | Which pole fails the gate: the high one (*with*) or the low one (*without*)? | `judge_scores.json` |
| 2b | Which questions and facets don't elicit? | `selection.json`, `trait_data_*/` |
| 3 | Which factors does each run keep? | `eliciting_questions.json`, `kept_traits.json` |
| 4 | How well do cosine and RP-LSH classify the factors, against chance? | `accuracy.csv` |
| 4b | The same models on persona-lsh's seven traits | `~/persona-lsh-output/<experiment>/accuracy.csv` |
| 5 | Which factors get confused, per run? | `predictions.json` |
| 5b | Which factor pairs get confused, over all runs? Is it agreeableness with honesty-humility? | `predictions.json` |
| 6 | Where in depth is the signal? | `layer_sweep.json` |
| 7 | How far apart are the factor directions, and how close is any question to being misclassified? | `all_*_diffs.pt` |
| 7a | Six directions or one axis? The literature's three predictions | `all_*_diffs.pt` |
| 7b | Do the vectors record the instruction or the behavior? (non-eliciting questions) | `all_*_diffs.pt` |
| 7c | Does the projection's size track the judge's score? | `all_*_diffs.pt` |
| 7d | How small can a digest be, and do 2, 3, 4 or 8 bits per projection beat 1? | `all_*_diffs.pt` |
| 7e | Which instruction pairs and factors are stable, and which vary? | `selection.json`, `all_*_diffs.pt` |
| 8 | Do the factor directions carry over within a model pair? | `pairs.json` |
| 9 | Do the two judges agree with each other on these rubrics? | `<run>/judges/*.json` |
| 9b | Does the choice of judge change the classification? | `<run>/judges/*.json`, `all_*_diffs.pt` |
| 10 | What did the cards do: time, power, kernel lines? | `~/gpu-telemetry/hexaco-lsh_*` |
| 11 | Interpretation | all of the above |

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

try:
    import torch   # sections 7 and 9b need it: the gpu env has it, and CPU torch is enough elsewhere
except ImportError:
    torch = None

OUT_ROOT = Path(os.environ.get("HEXACO_LSH_OUTPUT", Path.home() / "hexaco-lsh-output"))
EXPERIMENT = "latest"   # "latest": newest with an experiment.json | a directory name, e.g. "rand40-t256-s68853742"
if EXPERIMENT == "latest":
    _found = sorted(OUT_ROOT.glob("*/experiment.json"), key=lambda p: json.loads(p.read_text()).get("created", ""))
    assert _found, f"no experiment with an experiment.json under {OUT_ROOT}: copy it from the rig, or set HEXACO_LSH_OUTPUT"
    EXPERIMENT = _found[-1].parent.name
OUT_DIR = OUT_ROOT / EXPERIMENT
# The design as hexaco-lsh.ipynb recorded it (seed, split, framing, token budgets, judges, commits).
DESIGN = json.loads((OUT_DIR / "experiment.json").read_text()) if (OUT_DIR / "experiment.json").exists() else {}
JUDGES = list(DESIGN.get("judges", {"gemma-qat": False, "gpt-oss": "low"}))
TELEMETRY_ROOT = Path(os.environ.get("GPU_TELEMETRY", Path.home() / "gpu-telemetry"))
TELEMETRY_DIR = None    # None: the newest hexaco-lsh_* with a capture step. Set it to read another run's telemetry
_where = ([Path(os.environ["PERSONA_LSH"])] if "PERSONA_LSH" in os.environ
          else [Path.cwd(), *Path.cwd().parents, Path.home() / "persona-lsh", Path.home() / "persona-lsh" / "persona_vectors"])
PERSONA_LSH = next((p for p in _where if (p / "demo" / "expanded_analyze.py").is_file()), None)
TRAIT_ROOT = PERSONA_LSH / "experiments" if PERSONA_LSH else None   # the questions and their facets (2b)
# persona-lsh's seven traits on the same models (4b). "latest": its newest experiment with an experiment.json.
PERSONA_OUT_ROOT = Path(os.environ.get("PERSONA_LSH_OUTPUT", Path.home() / "persona-lsh-output"))
PERSONA_EXPERIMENT = "latest"

# As in hexaco-lsh.ipynb's parameters. The high pole is "with".
TRAITS = DESIGN.get("traits") or ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism",
                                  "honesty-humility"]
LETTER = {"openness": "O", "conscientiousness": "C", "extraversion": "E", "agreeableness": "A", "neuroticism": "N",
          "honesty-humility": "H"}
RUNS = ["qwen2.5-7b", "openreasoning-7b", "llama3.1-8b", "nemotron-nano-8b", "qwen3-8b", "qwen3-8b-think",
        "granite4.2-8b", "gemma4-12b", "bolt-e4b"]
N_QUESTIONS = 20
POS_THRESHOLD = DESIGN.get("gate", {}).get("with at least", 50)
NEG_THRESHOLD = DESIGN.get("gate", {}).get("without below", 50)
MIN_ELICITING = 4      # replaced by kept_traits.json's value when that file exists

SAVE_FIGS = True       # PNGs into OUT_DIR / "figures"
FIG_DIR = OUT_DIR / "figures"
print("outputs:", OUT_DIR, "| persona-lsh clone:", PERSONA_LSH, "| torch:", torch.__version__ if torch else "none")
if DESIGN:
    print("recorded design:", {k: v for k, v in DESIGN.items() if k not in ("traits", "commits")})

### Chart style

One palette for every figure: categorical slots in a fixed order (blue, orange, aqua, yellow, magenta, green), a
one-hue blue ramp for counts, and blue-gray-red for signed cosines. Identity never rests on color alone: series
also differ in marker or line style, and every chart has a legend or direct labels and a printed table.

In [ ]:
SURFACE, INK, INK2, MUTED, GRID, BASELINE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE, ORANGE = SLOTS[0], SLOTS[1]
SERIOUS = "#ec835a"   # status color: kernel errors, always with a text label
SEQ = LinearSegmentedColormap.from_list("seq", ["#f0efec", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#104281", "#5598e7", "#f0efec", "#ec8a89", "#a52a29"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2, "figure.dpi": 110})


def save(fig, name):
    if SAVE_FIGS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)


def ink_on(value, vmax):
    # Text color for a heatmap cell: white on the dark half of the ramp.
    return "white" if value > 0.55 * vmax else INK


def pct(v):
    return "" if v is None or pd.isna(v) else f"{v:.0%}"

## 0. What's in `OUT_DIR`

In [ ]:
def read_json(path, default=None):
    return json.loads(path.read_text()) if path.exists() else default


DATA = {}
for run in RUNS:
    d = OUT_DIR / run
    if (d / "run_info.json").exists():
        DATA[run] = {"info": read_json(d / "run_info.json"), "scores": read_json(d / "judge_scores.json", {}),
                     "elic": read_json(d / "eliciting_questions.json", {}),
                     "responses": read_json(d / "responses.json", {}),
                     # the drawn questions and framings; None for the published design
                     "picks": (read_json(d / "selection.json") or {}).get("traits")}

kept_file = read_json(OUT_DIR / "kept_traits.json")
if kept_file:
    MIN_ELICITING = kept_file["min eliciting"]
    KEPT = {label: v["kept"] for label, v in kept_file["runs"].items()}
    KEPT_COUNTS = {label: v.get("eliciting", {}) for label, v in kept_file["runs"].items()}
else:
    KEPT = {run: [t for t in TRAITS if len(d["elic"].get(t, [])) >= MIN_ELICITING] for run, d in DATA.items()}

accuracy_path = OUT_DIR / "accuracy.csv"
ACCURACY = pd.read_csv(accuracy_path) if accuracy_path.exists() else None
SWEEP = read_json(OUT_DIR / "layer_sweep.json")
PREDICTIONS = read_json(OUT_DIR / "predictions.json")
PAIR_RESULTS = read_json(OUT_DIR / "pairs.json")

stale = []
if not kept_file:
    stale.append("kept_traits.json")
if ACCURACY is not None and "check" not in ACCURACY:
    ACCURACY["check"] = "cv"   # files from before the held-out check
if ACCURACY is not None and "chance" not in ACCURACY:
    stale.append("accuracy.csv (no chance column)")
if SWEEP and not all(isinstance(v, dict) and "curve" in v for v in SWEEP.values()):
    stale.append("layer_sweep.json (old format)")
    SWEEP = None
for name, value in (("predictions.json", PREDICTIONS), ("pairs.json", PAIR_RESULTS)):
    if value is None:
        stale.append(name)
if stale:
    print("Missing or from before the per-trait filter, so rerun hexaco-lsh.ipynb (sections 5-7):", ", ".join(stale))

files = pd.DataFrame([{
    "run": run, "dtype": d["info"].get("dtype"), "traits captured": sum((OUT_DIR / run / f"all_{t}_diffs.pt").exists()
                                                                         for t in TRAITS),
    "traits judged": len(d["scores"]), "eliciting file": bool(d["elic"]),
    "kept (>= %d eliciting)" % MIN_ELICITING: len(KEPT.get(run, []))} for run, d in DATA.items()])
print(files.to_string(index=False))
print("not run:", [r for r in RUNS if r not in DATA])

## 1. Capture cost, and whether replies fit their budget

From `run_info.json` (seconds and generation speed per factor) and `responses.json` (reply lengths). A reply that
reaches `max_new_tokens` was cut off; it is still judged, on what it got to say.

In [ ]:
cost = []
for run, d in DATA.items():
    info, cap = d["info"], d["info"].get("max_new_tokens")
    lengths = [r[f"{c}_tokens"] for rows in d["responses"].values() for r in rows for c in ("with", "without")]
    traits = info.get("traits", {})
    cost.append({
        "run": run, "dtype": info.get("dtype"), "max new tokens": cap,
        "minutes": sum(t["seconds"] for t in traits.values()) / 60,
        "gen tok/s": np.mean([t["gen tok/s"] for t in traits.values()]) if traits else np.nan,
        "mean reply tokens": np.mean(lengths) if lengths else np.nan,
        "replies at the cap": (np.mean([n >= cap for n in lengths]) if lengths and cap else np.nan),
        "never finished thinking": sum(t.get("replies that never finished thinking", 0) for t in traits.values())
                                   if any("replies that never finished thinking" in t for t in traits.values()) else None,
        "non-finite captures": sum(t.get("non-finite captures", 0) for t in traits.values())})
COST = pd.DataFrame(cost).astype({"never finished thinking": "Int64"})
print(COST.to_string(index=False, na_rep="-", formatters={
    "minutes": "{:.0f}".format, "gen tok/s": "{:.1f}".format, "mean reply tokens": "{:.0f}".format,
    "replies at the cap": pct}))
print("\nThe minutes cover the traits captured in the last session (an earlier session's traits keep their own seconds).")

fig, ax = plt.subplots(figsize=(7, 0.35 * len(COST) + 0.9))
y = np.arange(len(COST))[::-1]
ax.barh(y, COST["minutes"], color=BLUE, height=0.6)
for yi, (m, capped) in zip(y, zip(COST["minutes"], COST["replies at the cap"])):
    ax.text(m, yi, f"  {m:.0f} min" + (f", {capped:.0%} at the token cap" if capped and capped > 0.2 else ""),
            va="center", color=INK2, fontsize=8)
ax.set_yticks(y, COST["run"])
ax.set_xlabel(f"capture minutes, all {len(TRAITS)} factors (generation + hidden-state read)")
ax.set_title("Capture time per run")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, COST["minutes"].max() * 1.45)
save(fig, "1_capture_minutes")
plt.show()

## 2. Which pole fails the gate

The gate keeps a question for a factor only if the judge scores the `with` reply (the high-pole instruction) at
50 or more **and** the `without` reply (the low-pole instruction) below 50. Each of a run's 240 question pairs
gets one outcome, checked in this order:

| Outcome | Meaning here |
|---|---|
| judge error | a judge call failed |
| no answer | a thinking reply never finished (no thinking runs in the first experiment) |
| refusal | the judge marked a reply as a refusal. Expected at the low poles of agreeableness and honesty-humility, which ask for hostile or dishonest replies |
| `with` below 50 | the **high pole** wasn't taken on. For neuroticism that is the departure from these models' default; for the other five the high pole is near the default |
| `without` 50+ | the **low pole** wasn't taken on: told to be careless, cold or boastful, the model still reads as the trait. For C, A and H this is the instructed departure, the reverse of persona-lsh |
| eliciting | passes the gate |

The mean scores under the figure say how far apart the two poles land when both are scored.

In [ ]:
OUTCOMES = ["eliciting", "with below 50", "without 50+", "refusal", "no answer", "judge error"]


def outcome(s):
    p, n = s["pos_score"], s["neg_score"]
    if any(str(x).startswith("ERROR") for x in (p, n)):
        return "judge error"
    if "NO_ANSWER" in (p, n):
        return "no answer"
    if "REFUSAL" in (p, n):
        return "refusal"
    if p < POS_THRESHOLD:
        return "with below 50"
    if n >= NEG_THRESHOLD:
        return "without 50+"
    return "eliciting"


GATE = pd.DataFrame([{"run": run, "trait": t, "qid": s["qid"], "outcome": outcome(s),
                      "pos": s["pos_score"] if isinstance(s["pos_score"], int) else np.nan,
                      "neg": s["neg_score"] if isinstance(s["neg_score"], int) else np.nan}
                     for run, d in DATA.items() for t in TRAITS for s in d["scores"].get(t, [])])

# The gate here must agree with eliciting_questions.json.
for run, d in DATA.items():
    for t in TRAITS:
        mine = sorted(GATE.query("run == @run and trait == @t and outcome == 'eliciting'")["qid"])
        if t in d["elic"] and mine != sorted(d["elic"][t]):
            print(f"MISMATCH {run}/{t}: {mine} vs {d['elic'][t]}")


def stacked(ax, table, title):
    table = table.reindex(columns=OUTCOMES, fill_value=0)
    y = np.arange(len(table))[::-1]
    left = np.zeros(len(table))
    for i, o in enumerate(OUTCOMES):
        ax.barh(y, table[o], left=left, color=SLOTS[i], height=0.65, edgecolor=SURFACE, linewidth=1, label=o)
        left += table[o].to_numpy()
    for yi, n in zip(y, table["eliciting"]):
        if n:
            ax.text(n / 2, yi, str(n), ha="center", va="center", color="white", fontsize=8)
    ax.set_yticks(y, table.index)
    ax.set_title(title)
    ax.grid(axis="y", visible=False)


by_run = GATE.groupby(["run", "outcome"]).size().unstack(fill_value=0).reindex(list(DATA))
by_trait = GATE.groupby(["trait", "outcome"]).size().unstack(fill_value=0).reindex(TRAITS)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
per_run = GATE.groupby("run").size()
stacked(axes[0], by_run, f"Gate outcome per run ({per_run.max()} question pairs each)")
stacked(axes[1], by_trait, f"Gate outcome per trait ({GATE.groupby(['run', 'trait']).size().max()} "
                                    f"questions x {len(DATA)} runs)")
axes[0].set_xlabel("question pairs")
axes[1].set_xlabel("question pairs")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.1))
fig.tight_layout()
save(fig, "2_gate_outcomes")
plt.show()

print("Per run and trait (eliciting / with below 50 / without 50+ / refusal / no answer / error):")
cell = GATE.groupby(["run", "trait", "outcome"]).size().unstack(fill_value=0).reindex(columns=OUTCOMES, fill_value=0)
print(cell.apply(lambda r: "/".join(map(str, r)), axis=1).unstack().reindex(index=list(DATA), columns=TRAITS).to_string())
print("\nMean judge score, with / without (integer scores only):")
means = GATE.groupby(["run", "trait"])[["pos", "neg"]].mean()
print(means.apply(lambda r: f"{r.pos:.0f} / {r.neg:.0f}", axis=1).unstack().reindex(index=list(DATA), columns=TRAITS).to_string())

### 2b. Which questions and facets don't elicit

The questions and instruction pairs in `trait_data_*/` are candidates that nobody has reviewed
(`../PLAN.md` section 9: read them before the run, or after a first run shows which ones don't elicit). Every run
of an experiment drawn by experiment gets the same 40 questions per factor with the same instruction pair, so a
question that fails in most runs points at the question or its pair, and one that fails in a single run points
at that model.

- **By facet:** the share of question pairs passing the gate, per factor facet and run (each question's `basis`
  in the trait file; honesty-humility has four facets, the Big Five six each).
- **By question:** the questions that fail in at least `FAIL_RUNS` runs, with their facet, instruction pair and
  the outcome in each run.

In [ ]:
FAIL_RUNS = 3            # list a question when it fails the gate in at least this many runs
SHORT = {"eliciting": "ok", "with below 50": "high pole", "without 50+": "low pole", "refusal": "refusal",
         "no answer": "no answer", "judge error": "error"}

POOLS = {}
if TRAIT_ROOT:
    for source in ("extract", "eval"):
        POOLS[source] = {t: read_json(TRAIT_ROOT / f"trait_data_{source}" / f"{t}.json") for t in TRAITS}
have_pools = bool(POOLS) and all(v for pool in POOLS.values() for v in pool.values())
drawn = {run: d for run, d in DATA.items() if d["picks"]}
if not have_pools:
    print("No trait data found (experiments/trait_data_extract and trait_data_eval): set PERSONA_LSH.")
elif not drawn:
    print("No selection.json: this section needs a drawn experiment.")
else:
    rows = []
    for run, d in drawn.items():
        for t in TRAITS:
            outcomes = {s["qid"]: outcome(s) for s in d["scores"].get(t, [])}
            for qid, p in enumerate(d["picks"][t]):
                if qid not in outcomes:
                    continue
                pool = POOLS[p["source"]][t]
                basis = (pool.get("basis") or [{}] * len(pool["questions"]))[p["index"]]
                rows.append({"run": run, "factor": t, "qid": qid, "question": f"{p['source']} {p['index']}",
                             "facet": basis.get("facet", "?"), "pair": p["framing"], "split": p["split"],
                             "outcome": outcomes[qid], "text": pool["questions"][p["index"]]})
    Q = pd.DataFrame(rows)
    Q["ok"] = Q["outcome"] == "eliciting"
    same_draw = Q.groupby(["factor", "qid"])["question"].nunique().max() == 1

    facets = {t: list(dict.fromkeys(Q[Q["factor"] == t]["facet"])) for t in TRAITS}
    share = Q.groupby(["factor", "facet", "run"])["ok"].mean().unstack().reindex(columns=list(drawn))
    asked = Q.groupby(["factor", "facet"])["qid"].nunique()
    cols = 3
    fig, axes = plt.subplots(-(-len(TRAITS) // cols), cols, figsize=(4.6 * cols, 2.5 * -(-len(TRAITS) // cols)),
                             squeeze=False)
    for ax, t in zip(axes.flat, TRAITS):
        m = share.loc[t].reindex(facets[t])
        ax.imshow(m.to_numpy(dtype=float), cmap=SEQ, vmin=0, vmax=1, aspect="auto")
        for a in range(m.shape[0]):
            for b in range(m.shape[1]):
                v = m.iat[a, b]
                if not pd.isna(v):
                    ax.text(b, a, f"{v:.0%}", ha="center", va="center", fontsize=7, color=ink_on(v, 1))
        ax.set_xticks(range(m.shape[1]), [r.split("-")[0][:8] for r in m.columns], fontsize=7)
        ax.set_yticks(range(m.shape[0]), [f"{f} ({asked.loc[(t, f)]})" for f in m.index], fontsize=7)
        ax.grid(False)
        ax.set_title(t, fontsize=9)
    for ax in list(axes.flat)[len(TRAITS):]:
        ax.set_visible(False)
    fig.suptitle("Share of question pairs passing the gate, by facet (questions asked) and run", x=0.01, ha="left",
                 fontweight="bold")
    fig.tight_layout()
    save(fig, "2b_facets")
    plt.show()

    pooled = Q.groupby(["factor", "facet"], sort=False)["ok"].agg(["mean", "size"])
    print("Facets from weakest, pooled over runs (share eliciting, question pairs):")
    print(pooled.sort_values("mean").head(10).to_string(formatters={"mean": pct}))

    if same_draw:
        fails = Q[~Q["ok"]].groupby(["factor", "qid"])["run"].nunique()
        listed = fails[fails >= FAIL_RUNS].reset_index(name="n")
        listed = listed.sort_values(["factor", "n", "qid"], key=lambda c: c.map(TRAITS.index) if c.name == "factor" else
                                    -c if c.name == "n" else c)
        listed = listed.set_index(["factor", "qid"])["n"]
        print(f"\nQuestions failing the gate in {FAIL_RUNS} or more of {len(drawn)} runs: {len(listed)} of "
              f"{Q.groupby(['factor', 'qid']).ngroups}")
        out = []
        for (t, qid), n in listed.items():
            g = Q[(Q["factor"] == t) & (Q["qid"] == qid)].set_index("run")
            first = g.iloc[0]
            out.append({"factor": LETTER.get(t, t), "qid": qid, "question": first["question"], "facet": first["facet"],
                        "pair": first["pair"], "fails": n,
                        **{r.split("-")[0][:8]: SHORT[g.loc[r, "outcome"]] if r in g.index else "" for r in drawn},
                        "text": first["text"][:70]})
        if out:
            print(pd.DataFrame(out).to_string(index=False))
        by_pair = Q.groupby(["factor", "pair"])["ok"].mean().unstack().reindex(TRAITS)
        print("\nShare eliciting by factor and instruction pair (section 7e has the directions):")
        print(by_pair.to_string(float_format=lambda v: f"{v:.0%}"))
    else:
        print("\nEach run drew its own questions (draw by run): no per-question list.")

## 3. Which factors each run keeps

Eliciting questions per run and factor, out of 40. Hatched cells are below `MIN_ELICITING` eliciting test
questions and are left out of that run's classification, layer sweep and pairs. Chance accuracy for a run is
1 / its kept factors.

In [ ]:
counts = pd.DataFrame({run: {t: len(d["elic"].get(t, [])) for t in TRAITS} for run, d in DATA.items()}).T
N_ASKED = max((len(s) for d in DATA.values() for s in d["scores"].values()), default=N_QUESTIONS)
fig, ax = plt.subplots(figsize=(8, 0.42 * len(counts) + 1.2))
ax.imshow(counts.to_numpy(), cmap=SEQ, vmin=0, vmax=N_ASKED, aspect="auto")
for i, run in enumerate(counts.index):
    for j, t in enumerate(TRAITS):
        n = counts.loc[run, t]
        dropped = t not in KEPT.get(run, [])   # a drawn design keeps by test-split count, not this total
        if dropped:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, hatch="///", edgecolor=MUTED, linewidth=0))
        ax.text(j, i, n, ha="center", va="center", fontsize=9, color=MUTED if dropped else ink_on(n, N_ASKED),
                fontweight="normal" if dropped else "bold")
ax.set_xticks(range(len(TRAITS)), TRAITS, rotation=30, ha="right")
ax.set_yticks(range(len(counts)), [f"{r}  ({len(KEPT.get(r, []))} kept)" for r in counts.index])
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title(f"Eliciting questions of {N_ASKED}; hatched = under {MIN_ELICITING} "
             f"{'test ' if any(d['picks'] for d in DATA.values()) else ''}eliciting, dropped for that run")
save(fig, "3_eliciting")
plt.show()

print(counts.assign(kept=[len(KEPT.get(r, [])) for r in counts.index],
                    chance=[pct(1 / len(KEPT[r])) if KEPT.get(r) else "-" for r in counts.index]).to_string())
print("\nPer trait, runs keeping it:", {t: sum(t in KEPT.get(r, []) for r in counts.index) for t in TRAITS})

## 4. Classification accuracy against chance

3-fold cross-validation over each run's kept factors (persona-lsh's `expanded_analyze` procedure), with cosine and
RP-LSH at the largest bit count. The gray bar marks chance for that run. The hollow circle is the held-out check:
means from the test set classify the eval set.

In [ ]:
if ACCURACY is None:
    print("No accuracy.csv: run hexaco-lsh.ipynb section 5.")
else:
    rp_cols = sorted((c for c in ACCURACY if c.startswith("rp ")), key=lambda c: int(c.split()[1]))
    RP = rp_cols[-1]
    every = ACCURACY[ACCURACY["strategy"].isin(["single", "multi", "all"])].copy()
    done = every[every["check"] == "cv"].copy()
    held = every[every["check"] == "held-out"]
    skipped = ACCURACY[~ACCURACY["strategy"].isin(["single", "multi", "all"])]
    if "chance" not in done:
        done["chance"] = np.nan
    labels = list(dict.fromkeys(done["run"]))
    fig, axes = plt.subplots(1, 3, figsize=(12, 0.38 * len(labels) + 1.4), sharey=True)
    for ax, strategy in zip(axes, ["single", "multi", "all"]):
        sub = done[done["strategy"] == strategy].set_index("run").reindex(labels)
        y = np.arange(len(labels))[::-1]
        ax.scatter(sub["chance"], y, marker="|", s=160, color=MUTED, linewidths=2, label="chance", zorder=2)
        ax.scatter(sub["cosine"], y + 0.15, marker="o", s=42, color=BLUE, label="cosine", zorder=3)
        ax.scatter(sub[RP], y - 0.15, marker="D", s=26, color=ORANGE, label=f"RP-LSH {RP.split()[1]} bits", zorder=4)
        if len(held):
            h = held[held["strategy"] == strategy].set_index("run").reindex(labels)
            ax.scatter(h["cosine"], y + 0.15, marker="o", s=70, facecolors="none", edgecolors=SLOTS[2], linewidths=1.5,
                       label="cosine, held-out eval set", zorder=5)
        ax.set_xlim(0, 1.04)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax.set_title(f"{strategy}")
        ax.grid(axis="y", visible=False)
        ax.set_yticks(y, labels)
    fig.suptitle("Accuracy per run and layer strategy: 3-fold CV, and the held-out eval set if drawn", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=4, bbox_to_anchor=(0.99, 1.0))
    save(fig, "4_accuracy")
    plt.show()

    # Accuracy above chance, scaled so 0 = chance and 1 = perfect.
    every["cosine lift"] = (every["cosine"] - every["chance"]) / (1 - every["chance"])
    every["rp lift"] = (every[RP] - every["chance"]) / (1 - every["chance"])
    every["n"] = every["n"].astype("Int64")
    cols = [c for c in ["run", "strategy", "check", "traits", "chance", "cosine", *rp_cols, "cosine lift", "rp lift", "n"]
            if c in every]
    print(every[cols].to_string(index=False, na_rep="", formatters={
        c: pct for c in cols if c not in ("run", "strategy", "check", "traits", "n")}))
    if len(skipped):
        print("\nskipped:", "; ".join(f"{r.run}: {r.strategy}" for r in skipped.itertuples()))

### 4b. The same models on persona-lsh's seven traits

`hexaco-lsh.ipynb` is `persona-lsh.ipynb` with the traits swapped, so a run here and the run of the same name in
a persona-lsh experiment differ in the traits (and their number: chance is 1/6 here) and nothing else, when the
designs match. The table puts the `single` strategy side by side, as accuracy and as lift (0 = chance, 1 =
perfect), which is the fair column when the two keep different numbers of traits. Needs
`~/persona-lsh-output/<experiment>/accuracy.csv`; skipped without it.

In [ ]:
def lift(acc, chance):
    return (acc - chance) / (1 - chance)


p_exp = PERSONA_EXPERIMENT
if p_exp == "latest":
    _found = sorted(PERSONA_OUT_ROOT.glob("*/experiment.json"), key=lambda p: json.loads(p.read_text()).get("created", ""))
    p_exp = _found[-1].parent.name if _found else None
p_acc = PERSONA_OUT_ROOT / p_exp / "accuracy.csv" if p_exp else None
if ACCURACY is None or not p_acc or not p_acc.exists():
    print("No persona-lsh accuracy.csv under", PERSONA_OUT_ROOT, "(or none here): nothing to compare.")
else:
    p_design = read_json(PERSONA_OUT_ROOT / p_exp / "experiment.json", {})
    keys = ("split", "framing", "max new tokens", "gate", "gate by")
    differs = {k: {"persona-lsh": p_design.get(k), "here": DESIGN.get(k)} for k in keys if p_design.get(k) != DESIGN.get(k)}
    print("persona-lsh experiment:", p_exp, "| design differences:", differs or "none")
    theirs = pd.read_csv(p_acc)
    if "check" not in theirs:
        theirs["check"] = "cv"
    rp = [c for c in ACCURACY if c.startswith("rp ") and c in theirs]
    rp = max(rp, key=lambda c: int(c.split()[1])) if rp else None
    rows = []
    for (run, check), mine in ACCURACY[ACCURACY["strategy"] == "single"].groupby(["run", "check"], sort=False):
        other = theirs[(theirs["run"] == run) & (theirs["strategy"] == "single") & (theirs["check"] == check)]
        if not len(other):
            continue
        a, b = mine.iloc[0], other.iloc[0]
        row = {"run": run, "check": check, "factors": int(a["traits"]), "traits": int(b["traits"]),
               "cosine, factors": a["cosine"], "cosine, traits": b["cosine"],
               "lift, factors": lift(a["cosine"], a["chance"]), "lift, traits": lift(b["cosine"], b["chance"])}
        if rp:
            row |= {f"{rp}, factors": a[rp], f"{rp}, traits": b[rp]}
        rows.append(row)
    if rows:
        VS = pd.DataFrame(rows)
        print(VS.to_string(index=False, formatters={c: pct for c in VS if c not in ("run", "check", "factors", "traits")}))
        gap = VS["cosine, traits"] - VS["cosine, factors"]
        print(f"\npersona traits minus factors, cosine: mean {gap.mean():+.1%}, range {gap.min():+.1%} to {gap.max():+.1%}"
              f" over {len(VS)} run-by-check rows")
    else:
        print("No run of the same name in both experiments.")

## 5. Confusions

From `predictions.json` (`single` strategy, every held-out question, largest RP bit count): rows are the true
factor, columns the prediction. Only runs with at least one mistake are drawn.

In [ ]:
def confusion(preds, traits, key):
    m = pd.DataFrame(0, index=traits, columns=traits)
    for p in preds:
        m.loc[p["true"], p[key]] += 1
    return m


if not PREDICTIONS:
    print("No predictions.json: run hexaco-lsh.ipynb section 5.")
else:
    shown = []
    for label, by_strategy in [(f"{l}{s}", {"single": p[f"single{s}"]})
                               for l, p in PREDICTIONS["runs"].items() for s in ("", " held-out") if f"single{s}" in p]:
        preds = by_strategy.get("single", [])
        traits = KEPT.get(label.removesuffix(" held-out")) or sorted({p["true"] for p in preds}, key=TRAITS.index)
        wrong = {k: [p for p in preds if p[k] != p["true"]] for k in ("cosine_pred", "rp_pred")}
        print(f"{label:26} single: cosine {len(wrong['cosine_pred'])} wrong, rp {len(wrong['rp_pred'])} wrong of {len(preds)}"
              + "".join(f" | {k[:-5]}: " + ", ".join(f"{p['true']} q{p['qid']} -> {p[k]}" for p in w)
                        for k, w in wrong.items() if w))
        for key in ("cosine_pred", "rp_pred"):
            if wrong[key]:
                shown.append((label, key, confusion(preds, traits, key)))
    if shown:
        cols = min(3, len(shown))
        rows = -(-len(shown) // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.8 * rows), squeeze=False)
        for ax, (label, key, m) in zip(axes.flat, shown):
            vmax = max(m.to_numpy().max(), 1)
            ax.imshow(m.to_numpy(), cmap=SEQ, vmin=0, vmax=vmax)
            for i in range(len(m)):
                for j in range(len(m)):
                    v = m.iat[i, j]
                    if v:
                        ax.text(j, i, v, ha="center", va="center", fontsize=8, color=ink_on(v, vmax),
                                fontweight="bold" if i != j else "normal")
            ax.set_xticks(range(len(m)), [t[:6] for t in m.columns], rotation=45, ha="right")
            ax.set_yticks(range(len(m)), m.index)
            ax.set_xlabel("predicted")
            ax.grid(False)
            ax.set_title(f"{label}, {'cosine' if key == 'cosine_pred' else 'RP-LSH'}")
        for ax in list(axes.flat)[len(shown):]:
            ax.set_visible(False)
        fig.tight_layout()
        save(fig, "5_confusions")
        plt.show()
    else:
        print("\nEvery question classified correctly at the single default layer (cross-validation and held-out): "
              "no confusion matrix to draw.")

### 5b. Which factor pairs get confused

Section 5's mistakes pooled over runs and both checks (cross-validation and held-out), at the single default
layer. The literature's predicted pair is agreeableness with honesty-humility (r = .41 between the IPIP-HEXACO
scales; HEXACO splits the Big Five's agreeableness between them). The self-report directions of `persona-hexaco`
confused every factor with neuroticism instead.

With 15 unordered pairs, a confusion spread evenly would put 1/15 of the mistakes on each: the last column is
each pair's share of all mistakes.

In [ ]:
if not PREDICTIONS:
    print("No predictions.json: run hexaco-lsh.ipynb section 5.")
else:
    pr = pd.DataFrame([p | {"run": run, "check": "held-out" if key.endswith("held-out") else "cv"}
                       for run, by in PREDICTIONS["runs"].items() for key, ps in by.items()
                       if key.startswith("single") for p in ps])
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
    for ax, (key, name) in zip(axes, (("cosine_pred", "cosine"), ("rp_pred", f"RP-LSH {PREDICTIONS.get('rp bits', '')}"))):
        m = pd.crosstab(pr["true"], pr[key]).reindex(index=TRAITS, columns=TRAITS, fill_value=0)
        off = m.to_numpy().copy()
        np.fill_diagonal(off, 0)
        vmax = max(off.max(), 1)
        ax.imshow(off, cmap=SEQ, vmin=0, vmax=vmax)
        for i in range(len(m)):
            for j in range(len(m)):
                v = m.iat[i, j]
                if v:
                    ax.text(j, i, v, ha="center", va="center", fontsize=8,
                            color=MUTED if i == j else ink_on(v, vmax), fontweight="normal" if i == j else "bold")
        ax.set_xticks(range(len(m)), [LETTER.get(t, t[:4]) for t in m.columns])
        ax.set_yticks(range(len(m)), m.index)
        ax.set_xlabel("predicted")
        ax.grid(False)
        ax.set_title(f"{name}: {int(off.sum())} wrong of {len(pr)}")
    fig.suptitle(f"Confusions pooled over {pr['run'].nunique()} runs, cv and held-out (diagonal in gray, not colored)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save(fig, "5b_pooled_confusions")
    plt.show()

    wrong = pr[pr["cosine_pred"] != pr["true"]]
    pair = wrong.apply(lambda r: " / ".join(sorted((r["true"], r["cosine_pred"]), key=TRAITS.index)), axis=1)
    n_pairs = len(TRAITS) * (len(TRAITS) - 1) // 2
    if len(wrong):
        table = pd.DataFrame({"mistakes": pair.value_counts(), "runs": wrong.groupby(pair)["run"].nunique()})
        table["share of mistakes"] = table["mistakes"] / len(wrong)
        print(f"Cosine mistakes by unordered pair ({len(wrong)} of {len(pr)}; an even spread is {1 / n_pairs:.0%} per pair):")
        print(table.sort_values("mistakes", ascending=False).to_string(formatters={"share of mistakes": pct}))
        ah = "agreeableness / honesty-humility"
        print(f"\n{ah}: {int(table['mistakes'].get(ah, 0))} of {len(wrong)} mistakes")
        direction = pd.crosstab(wrong["true"], wrong["cosine_pred"]).reindex(index=TRAITS, columns=TRAITS, fill_value=0)
        print("\nPer factor: its questions sent elsewhere (missed), and others' questions sent to it (attracted):")
        print(pd.DataFrame({"missed": direction.sum(axis=1), "attracted": direction.sum(axis=0),
                            "questions": pr.groupby("true").size().reindex(TRAITS)}).to_string())
    else:
        print("No cosine mistakes at the single default layer.")

## 6. Where in depth the signal sits

`single` strategy at every layer, on each run's kept factors. Depth is relative (layer / layers), so models of
different depth line up; the thin vertical line is persona-lsh's default layer (20/28 of depth). The y axis starts
well above chance (printed in each title) so the differences between layers are visible.

In [ ]:
if not SWEEP:
    print("No layer_sweep.json in the per-trait format: run hexaco-lsh.ipynb section 6.")
else:
    labels = list(SWEEP)
    cols = 4
    rows = -(-len(labels) // cols)
    lows = [min(min(v) for v in s["curve"].values()) for s in SWEEP.values()]
    floor = max(0.0, np.floor(min(lows) * 20) / 20 - 0.05)
    fig, axes = plt.subplots(rows, cols, figsize=(3.3 * cols, 2.5 * rows), sharex=True, sharey=True, squeeze=False)
    summary = []
    for ax, label in zip(axes.flat, labels):
        s = SWEEP[label]
        n = s["layers"]
        layers = sorted(int(l) for l in s["curve"])
        depth = [l / n for l in layers]
        cos = [s["curve"][str(l)][0] for l in layers]
        rp = [s["curve"][str(l)][1] for l in layers]
        ax.plot(depth, cos, color=BLUE, label="cosine")
        ax.plot(depth, rp, color=ORANGE, linestyle=(0, (3, 2)), label=f"RP-LSH {s['rp bits']}")
        ax.axvline(20 / 28, color=MUTED, linewidth=0.8)
        ax.set_title(f"{label}\n{len(s['traits'])} traits, chance {1 / len(s['traits']):.0%}", fontsize=9)
        ax.set_ylim(floor, 1.01)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        best = max(layers, key=lambda l: (cos[l - 1] + rp[l - 1], -abs(l / n - 0.7)))
        band = [l for l, c in zip(layers, cos) if c >= 0.99]
        summary.append({"run": label, "layers": n, "traits": len(s["traits"]),
                        "layer 1 cosine": cos[0], "min cosine": min(cos), "at layer": layers[int(np.argmin(cos))],
                        "best layer": best, "best depth": best / n, "first layer >= 99%": band[0] if band else None,
                        "depth": band[0] / n if band else None})
    for ax in list(axes.flat)[len(labels):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("relative depth")
    axes[0, 0].legend(loc="lower right", fontsize=8)
    fig.suptitle("Single-layer accuracy across depth", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save(fig, "6_layer_sweep")
    plt.show()
    SWEEP_SUMMARY = pd.DataFrame(summary)
    print(SWEEP_SUMMARY.to_string(index=False, na_rep="-", formatters={
        c: pct for c in ("layer 1 cosine", "min cosine", "best depth", "depth")}))

## 7. Geometry: how far apart the factors are

Needs the `.pt` vectors. For each run at persona-lsh's default layer:

- **Factor-to-factor cosine** between the mean vectors (each the mean of that factor's eliciting
  `with − without` differences, so it points from the low pole to the high pole). Off-diagonal values near 1
  mean two factors share a direction; near 0, they're separable; negative, one's high pole lies toward the
  other's low pole.
- **Margin** for every eliciting question: cosine to its own factor's mean (leaving that question out) minus the
  largest cosine to another kept factor's mean. Below 0, nearest-mean cosine misclassifies it.

In [ ]:
def single_default(n_layers):
    return max(1, round(n_layers * 20 / 28))


GEOMETRY, MARGINS = {}, []
if torch is None:
    print("No torch in this kernel: section 7 needs the gpu env.")
else:
    for run in DATA:
        traits = KEPT.get(run, [])
        if len(traits) < 2 or not all((OUT_DIR / run / f"all_{t}_diffs.pt").exists() for t in traits):
            continue
        diffs = {t: torch.load(OUT_DIR / run / f"all_{t}_diffs.pt").float() for t in traits}
        layer = single_default(diffs[traits[0]].shape[1] - 1)
        vecs = {t: diffs[t][DATA[run]["elic"][t], layer] for t in traits}
        means = {t: v.mean(0) for t, v in vecs.items()}
        unit = lambda x: x / x.norm(dim=-1, keepdim=True)
        M = torch.stack([unit(means[t]) for t in traits])
        GEOMETRY[run] = pd.DataFrame((M @ M.T).numpy(), index=traits, columns=traits)
        for t in traits:
            v = vecs[t]
            for i in range(len(v)):
                own = unit((means[t] * len(v) - v[i]) / (len(v) - 1))   # leave this question out
                others = max(float(unit(v[i]) @ unit(means[u])) for u in traits if u != t)
                MARGINS.append({"run": run, "trait": t, "qid": DATA[run]["elic"][t][i],
                                "margin": float(unit(v[i]) @ own) - others})

if GEOMETRY:
    cols = 4
    rows = -(-len(GEOMETRY) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.9 * cols, 3.4 * rows), squeeze=False)
    fig.subplots_adjust(wspace=0.55, hspace=0.45)
    for ax, (run, g) in zip(axes.flat, GEOMETRY.items()):
        im = ax.imshow(g.to_numpy(), cmap=DIV, vmin=-1, vmax=1)
        for i in range(len(g)):
            for j in range(len(g)):
                if i != j:
                    v = g.iat[i, j]
                    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7,
                            color="white" if abs(v) > 0.6 else INK)
        ax.set_xticks(range(len(g)), [t[:4] for t in g.columns], fontsize=7)
        ax.set_yticks(range(len(g)), [t[:6] for t in g.index], fontsize=7)
        ax.grid(False)
        ax.set_title(run, fontsize=9)
    for ax in list(axes.flat)[len(GEOMETRY):]:
        ax.set_visible(False)
    fig.colorbar(im, ax=axes, shrink=0.6, label="cosine between mean persona vectors")
    fig.suptitle("Trait-to-trait cosine at the default layer (kept traits)", x=0.01, ha="left", fontweight="bold")
    save(fig, "7_trait_cosines")
    plt.show()

    pairs_flat = pd.DataFrame([{"run": r, "a": a, "b": b, "cos": g.loc[a, b]} for r, g in GEOMETRY.items()
                               for i, a in enumerate(g.index) for b in g.index[i + 1:]])
    print("Most similar trait pairs per run:")
    for r, sub in pairs_flat.groupby("run", sort=False):
        top = sub.nlargest(3, "cos")
        print(f"  {r:18} " + "; ".join(f"{x.a}-{x.b} {x.cos:.2f}" for x in top.itertuples()))
    print("Mean cosine of each trait pair over the runs keeping both:")
    print(pairs_flat.groupby(["a", "b"])["cos"].agg(["mean", "count"]).sort_values("mean", ascending=False)
          .round(2).to_string())

if MARGINS:
    MARGIN = pd.DataFrame(MARGINS)
    runs = list(dict.fromkeys(MARGIN["run"]))
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(runs) + 1.2))
    rng = np.random.default_rng(0)
    for i, run in enumerate(runs):
        m = MARGIN[MARGIN["run"] == run]["margin"]
        y = len(runs) - 1 - i + rng.uniform(-0.18, 0.18, len(m))
        ax.scatter(m, y, s=10, color=np.where(m < 0, ORANGE, BLUE), alpha=0.8, linewidths=0)
    ax.axvline(0, color=INK2, linewidth=1)
    ax.set_yticks(range(len(runs))[::-1], runs)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("cosine to own trait (leave-one-out) − best other trait")
    ax.legend(handles=[Line2D([], [], marker="o", linestyle="", color=BLUE, label="classified right"),
                       Line2D([], [], marker="o", linestyle="", color=ORANGE, label="misclassified (margin < 0)")],
              loc="upper left", bbox_to_anchor=(0, -0.12), ncol=2)
    ax.set_title("Classification margin of each eliciting question, default layer")
    save(fig, "7_margins")
    plt.show()
    print(MARGIN.groupby("run", sort=False)["margin"].describe(percentiles=[0.05, 0.5])[["count", "min", "5%", "50%"]]
          .round(3).to_string())
    print("\nSmallest margins:")
    print(MARGIN.nsmallest(8, "margin").round(3).to_string(index=False))

### 7a. Six directions or one axis?

The self-report directions of `persona-hexaco` were mostly one axis: leakage between factors of 0.5-1.0, every
confused pair involving neuroticism, read as social desirability or agreement (`PLAN.md` section 5). These
directions come from instructed open replies instead. Three checks on section 7's cosines:

- **The literature's three predictions** (`PLAN.md` section 5 part B): N opposed to E, N opposed to C, and H
  related to A. Per run, the cosine and whether its sign is the predicted one.
- **Desirable-keyed cosines.** Flip neuroticism so that every direction points at the socially desirable pole
  (high O, C, E, A, H; low N). If one desirability axis carries the factors, every off-diagonal cosine is then
  positive and large. The mean and the smallest are printed per run.
- **Share of the first component.** The largest eigenvalue of the 6 x 6 cosine matrix, over 6: the share of the
  six unit directions' variance on one axis (not centered). Six orthogonal directions give 1/6 = 17%, one shared
  axis 100%. Its loadings say which factors are on it and with which sign.

One caution on reading a shared component as desirability. Every direction is *with − without*, and for five of
the six factors *with* is the pole the model takes by default. So a component common to all six can also be
"the instruction agrees with the default" against "the instruction asks for a departure". Neuroticism separates
the two readings: its high pole is undesirable **and** the departure, so both predict it loads with the opposite
sign, and this run alone can't tell them apart.

In [ ]:
PREDICTED = {("neuroticism", "extraversion"): -1, ("neuroticism", "conscientiousness"): -1,
             ("agreeableness", "honesty-humility"): +1}
DESIRABLE = {t: -1.0 if t == "neuroticism" else 1.0 for t in TRAITS}   # sign that points a direction at the desirable pole

if not GEOMETRY:
    print("No geometry (section 7 needs torch and the .pt vectors).")
else:
    rows, axis_rows, loadings = [], [], {}
    for run, g in GEOMETRY.items():
        for (a, b), sign in PREDICTED.items():
            if a in g.index and b in g.index:
                c = float(g.loc[a, b])
                rows.append({"run": run, "pair": f"{LETTER[a]}-{LETTER[b]}", "predicted": "+" if sign > 0 else "-",
                             "cosine": c, "as predicted": c * sign > 0})
        s = np.array([DESIRABLE.get(t, 1.0) for t in g.index])
        keyed = g.to_numpy() * np.outer(s, s)
        off = keyed[~np.eye(len(g), dtype=bool)]
        values, vectors = np.linalg.eigh(g.to_numpy())
        v = vectors[:, -1]
        v = v * (1 if (v * s).sum() >= 0 else -1)      # orient so the desirable poles load positive overall
        loadings[run] = pd.Series(v, index=g.index)
        axis_rows.append({"run": run, "factors": len(g), "first component": values[-1] / len(g),
                          "if orthogonal": 1 / len(g), "second": values[-2] / len(g),
                          "mean cosine, desirable-keyed": off.mean(), "min": off.min(),
                          "mean |cosine|": np.abs(off).mean()})
    PRED = pd.DataFrame(rows)
    AXIS = pd.DataFrame(axis_rows)

    full = [g.reindex(index=TRAITS, columns=TRAITS) for g in GEOMETRY.values()]
    stack = np.stack([g.to_numpy(dtype=float) for g in full])
    counted = (~np.isnan(stack)).sum(0)
    mean = np.where(counted > 0, np.nansum(stack, 0) / np.maximum(counted, 1), np.nan)
    sd = np.sqrt(np.where(counted > 0, np.nansum((stack - mean) ** 2, 0) / np.maximum(counted, 1), np.nan))
    same_sign = np.where(counted > 0, np.nansum(np.sign(stack) == np.sign(mean), 0), 0)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios": [1.15, 1]})
    ax = axes[0]
    im = ax.imshow(mean, cmap=DIV, vmin=-1, vmax=1)
    for i in range(len(TRAITS)):
        for j in range(len(TRAITS)):
            if i != j and not np.isnan(mean[i, j]):
                ax.text(j, i, f"{mean[i, j]:+.2f}\n±{sd[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(mean[i, j]) > 0.6 else INK)
    for (a, b), sign in PREDICTED.items():
        for i, j in ((TRAITS.index(a), TRAITS.index(b)), (TRAITS.index(b), TRAITS.index(a))):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=INK, linewidth=1.5))
    ax.set_xticks(range(len(TRAITS)), [LETTER.get(t, t[:4]) for t in TRAITS])
    ax.set_yticks(range(len(TRAITS)), TRAITS)
    ax.grid(False)
    ax.set_title(f"Mean cosine over {len(GEOMETRY)} runs, ± SD (outlined: predicted pairs)")
    fig.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1]
    y = np.arange(len(AXIS))[::-1]
    ax.barh(y, AXIS["first component"], color=BLUE, height=0.55, label="first component")
    ax.scatter(AXIS["if orthogonal"], y, marker="|", s=220, color=MUTED, linewidths=2, zorder=3, label="six orthogonal directions")
    for yi, v in zip(y, AXIS["first component"]):
        ax.text(v, yi, f"  {v:.0%}", va="center", fontsize=8, color=INK2)
    ax.set_yticks(y, AXIS["run"])
    ax.set_xlim(0, 1.05)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right", fontsize=8)
    ax.set_title("Share of the directions on one axis")
    fig.tight_layout()
    save(fig, "7a_one_axis")
    plt.show()

    print("The literature's predictions, per run:")
    print(PRED.pivot(index="run", columns="pair", values="cosine").reindex(list(GEOMETRY)).round(2).to_string())
    print("as predicted:", {p: f"{int(g['as predicted'].sum())} of {len(g)} runs" for p, g in PRED.groupby("pair", sort=False)})
    print("\nOne axis?")
    print(AXIS.to_string(index=False, formatters={c: pct for c in ("first component", "if orthogonal", "second")},
                         float_format=lambda v: f"{v:+.2f}"))
    print("\nLoadings on the first component (oriented so the desirable poles sum positive):")
    print(pd.DataFrame(loadings).T.reindex(columns=TRAITS).round(2).to_string())
    pairs_sign = pd.DataFrame(same_sign, index=TRAITS, columns=TRAITS)
    stable = [(TRAITS[i], TRAITS[j], mean[i, j]) for i in range(len(TRAITS)) for j in range(i + 1, len(TRAITS))
              if counted[i, j] == len(GEOMETRY) and pairs_sign.iat[i, j] == len(GEOMETRY)]
    print(f"\nPairs with the same sign in every run ({len(stable)} of {len(TRAITS) * (len(TRAITS) - 1) // 2}):",
          ", ".join(f"{LETTER[a]}-{LETTER[b]} {m:+.2f}" for a, b, m in sorted(stable, key=lambda x: -abs(x[2]))))

### 7b. Instruction or behavior? The questions that didn't elicit

The persona means come from eliciting questions only. Every other question still has a `with − without`
vector: the model got the same persona instruction but (per the judge) didn't act on it, refused, or never
answered. Classify those vectors against the eliciting means, at the default layer, over the run's kept traits.

- Accuracy near the eliciting level means the vector records **the instruction the model was given**, whether
  or not it acted on it. Then a digest flags what a model was asked to be (a system-prompt or persona
  injection), not only what it did.
- Accuracy near chance means the vector records **the behavior**, and a digest is evidence the trait was
  expressed.

These questions are out of sample: none of them is in the means. Section 7's eliciting margins are the
comparison.

In [ ]:
NONELIC = []
if torch is not None:
    unit = lambda x: x / x.norm(dim=-1, keepdim=True)
    for run in DATA:
        traits = KEPT.get(run, [])
        if len(traits) < 2 or not all((OUT_DIR / run / f"all_{t}_diffs.pt").exists() for t in traits):
            continue
        diffs = {t: torch.load(OUT_DIR / run / f"all_{t}_diffs.pt").float() for t in traits}
        layer = single_default(diffs[traits[0]].shape[1] - 1)
        elic = {t: DATA[run]["elic"][t] for t in traits}
        M = torch.stack([unit(diffs[t][elic[t], layer].mean(0)) for t in traits])
        for t in traits:
            outcomes = {s["qid"]: outcome(s) for s in DATA[run]["scores"][t]}
            for q in range(diffs[t].shape[0]):
                v = diffs[t][q, layer]
                if q in elic[t] or not torch.isfinite(v).all():
                    continue
                pred = traits[int((M @ unit(v)).argmax())]
                NONELIC.append({"run": run, "trait": t, "qid": q, "outcome": outcomes.get(q, "?"),
                                "predicted": pred, "correct": pred == t, "chance": 1 / len(traits)})

if NONELIC:
    NE = pd.DataFrame(NONELIC)
    by_outcome = NE.groupby("outcome").agg(n=("correct", "size"), accuracy=("correct", "mean"),
                                           chance=("chance", "mean")).reindex([o for o in OUTCOMES if o != "eliciting"]).dropna()
    by_run = NE.groupby("run", sort=False).agg(n=("correct", "size"), accuracy=("correct", "mean"), chance=("chance", "mean"))
    print("Non-eliciting questions, classified against the eliciting means (default layer, cosine):")
    print(by_outcome.to_string(formatters={"accuracy": pct, "chance": pct, "n": "{:.0f}".format}))
    print()
    print(by_run.to_string(formatters={"accuracy": pct, "chance": pct, "n": "{:.0f}".format}))
    print("\nBy run and trait (correct / n):")
    print(NE.groupby(["run", "trait"])["correct"].agg(lambda s: f"{s.sum()}/{len(s)}").unstack()
          .reindex(index=[r for r in DATA if r in set(NE["run"])], columns=TRAITS).fillna("").to_string())

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(by_outcome) + 1.2))
    y = np.arange(len(by_outcome))[::-1]
    ax.barh(y, by_outcome["accuracy"], color=BLUE, height=0.55)
    ax.scatter(by_outcome["chance"], y, marker="|", s=220, color=MUTED, linewidths=2, zorder=3, label="chance (mean)")
    for yi, (acc, n) in zip(y, zip(by_outcome["accuracy"], by_outcome["n"])):
        ax.text(acc, yi, f"  {acc:.0%} of {n:.0f}", va="center", fontsize=8, color=INK2)
    ax.set_yticks(y, by_outcome.index)
    ax.set_xlim(0, 1.15)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right", fontsize=8)
    ax.set_title("Non-eliciting questions classified as their instructed trait")
    save(fig, "7b_non_eliciting")
    plt.show()

### 7c. Does the projection track how strongly the trait shows?

For every question of a kept trait (eliciting or not), the projection of its `with − without` vector onto
its trait's unit persona direction, against the judge's score for the `with` reply. A Spearman ρ near 1 means
the projection's size grades the trait, the signal a magnitude bit would keep (persona-lsh README direction
8). Sign-only RP-LSH keeps the direction and drops it.

In [ ]:
PROJ = []
if torch is not None:
    unit = lambda x: x / x.norm(dim=-1, keepdim=True)
    for run in DATA:
        traits = KEPT.get(run, [])
        if len(traits) < 2 or not all((OUT_DIR / run / f"all_{t}_diffs.pt").exists() for t in traits):
            continue
        diffs = {t: torch.load(OUT_DIR / run / f"all_{t}_diffs.pt").float() for t in traits}
        layer = single_default(diffs[traits[0]].shape[1] - 1)
        for t in traits:
            u = unit(diffs[t][DATA[run]["elic"][t], layer].mean(0))
            for s in DATA[run]["scores"][t]:
                v = diffs[t][s["qid"], layer]
                if isinstance(s["pos_score"], int) and torch.isfinite(v).all():
                    PROJ.append({"run": run, "trait": t, "qid": s["qid"], "projection": float(v @ u),
                                 "with score": s["pos_score"], "eliciting": s["qid"] in DATA[run]["elic"][t]})

if PROJ:
    P = pd.DataFrame(PROJ)

    def spearman(g):
        return g["projection"].rank().corr(g["with score"].rank()) if len(g) >= 5 else np.nan

    RHO = P.groupby(["run", "trait"]).apply(spearman, include_groups=False).unstack()
    RHO = RHO.reindex(index=[r for r in DATA if r in RHO.index], columns=[t for t in TRAITS if t in RHO.columns])
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(RHO) + 1.2))
    ax.imshow(RHO.to_numpy(dtype=float), cmap=DIV, vmin=-1, vmax=1, aspect="auto")
    for a in range(RHO.shape[0]):
        for b in range(RHO.shape[1]):
            v = RHO.iat[a, b]
            ax.text(b, a, "" if pd.isna(v) else f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if not pd.isna(v) and abs(v) > 0.6 else INK)
    ax.set_xticks(range(RHO.shape[1]), RHO.columns, rotation=30, ha="right")
    ax.set_yticks(range(RHO.shape[0]), RHO.index)
    ax.grid(False)
    ax.set_title("Spearman ρ: projection on the persona direction vs judge score of the with reply")
    save(fig, "7c_projection_vs_score")
    plt.show()
    print(RHO.round(2).to_string())
    pooled = P.groupby("run", sort=False).apply(spearman, include_groups=False)
    print("\nPooled per run (all kept traits):", {r: round(v, 2) for r, v in pooled.items()})
    print("median ρ over run x trait cells:", round(float(np.nanmedian(RHO.to_numpy(dtype=float))), 2))

### 7d. Digest size, and bits per projection (README direction 8)

At persona-lsh's default layer, over each run's kept traits, every code gets the same budget of B bits:

| Code | What it keeps |
|---|---|
| **1 bit (RP-LSH)** | the sign of B random projections; a trait's reference is the sign digest of its mean vector, compared by Hamming distance. Today's persona-lsh. |
| **1 bit, centroid** | the same bits, but the reference is the mean of the trait's training digests (L1). Separates the reference from the code. |
| **b bits, angle** (b = 2, 3, 4, 8) | B/b projections of the *unit* vector, each as 1 sign bit + (b−1) bits of distance from the hyperplane, bucketed at quantiles of the training projections. Finer angle, no length. |
| **b bits, magnitude** | the same code on the *raw* vector, so the buckets also carry its length: how strongly the persona shows (7c). This is the version README direction 8 means, and it leaks more than the angle. |

Multi-bit codes are compared with each trait's centroid code by summed level difference (L1). Two checks: **leave-
one-out** over every eliciting question, and, in a drawn design, **held-out** (references from the test split,
the eval split classified). Each over 5 projection seeds; the 95% intervals resample questions (each question's
result averaged over seeds). The bucket edges come from the training questions without their labels (in
leave-one-out, including the question left out).

The per-run table gives the smallest digest at which today's RP-LSH reaches 98% and 99% (PLAN §2d).

Sizes run from 16 bits to 131072 (16 KiB), to find the bottom of each curve. Past the model's hidden size
(2560-4096) the projections stop being a compression; they're only there to find the floor.

**The floor of today's RP-LSH is exact cosine.** The share of sign bits on which two vectors differ tends to
their angle / π as the digest grows, so nearest digest by Hamming distance tends to nearest mean by cosine. That
error is computed directly (same questions, same leave-one-out and held-out rules), printed under each table
with the 1-bit curve's distance from it, and drawn as a line. The centroid codes use another rule (L1 to a mean
of codes) and can end above or below it.

**Why the excess over the floor should fall like bits^-0.5.** With B independent sign bits, the Hamming estimate
of an angle has noise proportional to 1/√B. A question is misread when that noise exceeds its margin, so each
question's error falls fast once √B passes 1/margin, and the pooled curve is the sum of those steps: with
near-ties spread evenly, the share still inside the noise is proportional to 1/√B. The printed slope checks it.

**3 bits** is one sign bit and two bits of distance (4 levels). If 2 bits help and 4 hurt, 3 may be the best
trade of precision against projections. A 3-bit code uses ⌊B/3⌋ projections, so it's up to 2 bits under budget.

**By magnitude.** The same errors again, split by how long the question's `with − without` vector is: thirds of
its norm among the eliciting questions of the same run and trait (so a third isn't one model or one trait). A
short vector is a weak persona shift. If the low third misses more, the errors are weak expressions, and if a
bigger digest closes that gap, the hash was the limit for them. Shown for today's RP-LSH and the two 2-bit codes.

In [ ]:
BUDGETS = [2 ** k for k in range(4, 18)]            # digest bits: 16 to 131072 (16 KiB)
DEPTHS = [2, 3, 4, 8]                               # bits per projection: 1 sign bit + (b-1) of distance; 3 = 4 distance levels
SEEDS = range(5)                                    # projection seeds
BOOT = 2000                                         # bootstrap resamples for the intervals
STRATA = ["low", "middle", "high"]                  # thirds of the vector's norm, within a run and trait
STRATA_CODES = ["1 bit (RP-LSH)", "2 bits, angle", "2 bits, magnitude", "3 bits, magnitude"]
CODES = ["1 bit (RP-LSH)", "1 bit, centroid"] + [f"{b} bits, {k}" for k in ("angle", "magnitude") for b in DEPTHS]


def edges(P, b):
    # Bucket edges of a b-bit code: 0 and +-quantiles of |P| (pooled over questions and planes), so the
    # first bit is the sign and the other b-1 bits say how far from the hyperplane.
    m = 2 ** (b - 1)
    a = P.abs().flatten()
    a = a[::-(-len(a) // 4_000_000)]   # torch.quantile takes at most 2^24 values: an even subsample is enough
    q = torch.quantile(a, torch.arange(1, m, dtype=P.dtype) / m) if m > 1 else P.new_zeros(0)
    return torch.cat([-q.flip(0), P.new_zeros(1), q])


def nearest_centroid(train, test, traits, loo):
    # Each test code's nearest trait centroid (mean of the trait's training codes), by L1. In leave-one-out,
    # test is train, and a question is left out of its own trait's centroid.
    sums = {t: train[t].sum(0) for t in traits}
    M = torch.stack([sums[t] / len(train[t]) for t in traits])
    out = {}
    for k, t in enumerate(traits):
        C = test[t]
        if not len(C):
            continue
        D = torch.cdist(C, M, p=1)
        if loo:
            D[:, k] = (C - (sums[t] - C) / (len(C) - 1)).abs().sum(1)
        out[t] = (D.argmin(1) == k).tolist()   # ties go to the first trait, as min() does in persona-lsh
    return out


def nearest_sign_of_mean(train, test, traits, loo):
    # Today's RP-LSH: a trait's digest is the sign of its mean vector's projections (the sign of the summed
    # projections: projection is linear), compared by Hamming distance.
    sums = {t: train[t].sum(0) for t in traits}
    ref = torch.stack([sums[t] > 0 for t in traits]).float()
    out = {}
    for k, t in enumerate(traits):
        C = test[t]
        if not len(C):
            continue
        bits = (C > 0).float()
        D = torch.cdist(bits, ref, p=1)
        if loo:
            D[:, k] = (bits - ((sums[t] - C) > 0).float()).abs().sum(1)
        out[t] = (D.argmin(1) == k).tolist()
    return out


def nearest_cosine(X, train_idx, test_idx, traits, loo):
    # Exact cosine to each trait's mean vector: what today's RP-LSH tends to as the digest grows (the share of
    # differing sign bits tends to angle / pi), so its error is the floor of the 1-bit curve.
    unit = lambda x: x / x.norm(dim=-1, keepdim=True)
    sums = {t: X[t][train_idx[t]].sum(0) for t in traits}
    M = torch.stack([unit(sums[t]) for t in traits])
    out = {}
    for k, t in enumerate(traits):
        V = X[t][test_idx[t]]
        if not len(V):
            continue
        S = unit(V) @ M.T
        if loo:
            S[:, k] = (unit(V) * unit(sums[t] - V)).sum(1)
        out[t] = (S.argmax(1) == k).tolist()
    return out


BITS_ROWS, NORM_ROWS, FLOOR_ROWS = [], [], []
if torch is not None:
    for run in DATA:
        traits = KEPT.get(run, [])
        if len(traits) < 2 or not all((OUT_DIR / run / f"all_{t}_diffs.pt").exists() for t in traits):
            continue
        diffs = {t: torch.load(OUT_DIR / run / f"all_{t}_diffs.pt").float() for t in traits}
        layer = single_default(diffs[traits[0]].shape[1] - 1)
        picks = DATA[run]["picks"]
        qids = {t: [q for q in DATA[run]["elic"][t] if torch.isfinite(diffs[t][q, layer]).all()] for t in traits}
        X = {t: diffs[t][qids[t], layer] for t in traits}
        for t in traits:   # magnitude strata: rank each eliciting question's norm among its run and trait
            norm = X[t].norm(dim=1)
            third = (norm.argsort().argsort() * 3 // max(len(norm), 1)).tolist()
            NORM_ROWS += [{"run": run, "trait": t, "qid": q, "norm": float(n), "relative norm": float(n / norm.median()),
                           "stratum": STRATA[k]} for q, n, k in zip(qids[t], norm, third)]
        checks = {"leave-one-out": {t: list(range(len(qids[t]))) for t in traits}}   # every eliciting question
        if picks:   # held-out: references from the test split, classify the eval split
            checks["held-out"] = {t: [i for i, q in enumerate(qids[t]) if picks[t][q]["split"] == "test"] for t in traits}
        for check, train_idx in checks.items():
            loo = check == "leave-one-out"
            test_idx = train_idx if loo else {t: sorted(set(range(len(qids[t]))) - set(train_idx[t])) for t in traits}
            for t, ok in nearest_cosine(X, train_idx, test_idx, traits, loo).items():
                FLOOR_ROWS += [{"run": run, "check": check, "trait": t, "qid": qids[t][i], "correct": c}
                               for i, c in zip(test_idx[t], ok)]
        for seed in SEEDS:
            R = torch.randn(max(BUDGETS), X[traits[0]].shape[1], generator=torch.Generator().manual_seed(seed))
            raw = {t: X[t] @ R.T for t in traits}                               # projections [n, planes]
            unit = {t: raw[t] / X[t].norm(dim=1, keepdim=True) for t in traits}
            for check, train_idx in checks.items():
                loo = check == "leave-one-out"
                test_idx = train_idx if loo else {t: sorted(set(range(len(qids[t]))) - set(train_idx[t])) for t in traits}
                for budget in BUDGETS:
                    results = {"1 bit (RP-LSH)": nearest_sign_of_mean(
                        {t: raw[t][train_idx[t], :budget] for t in traits},
                        {t: raw[t][test_idx[t], :budget] for t in traits}, traits, loo)}
                    for b, kind in [(1, "centroid")] + [(b, k) for k in ("angle", "magnitude") for b in DEPTHS]:
                        planes = budget // b
                        if planes < 1:
                            continue
                        P = unit if kind == "angle" else raw
                        e = edges(torch.cat([P[t][train_idx[t], :planes] for t in traits]), b)
                        code = {t: torch.bucketize(P[t][:, :planes].contiguous(), e).float() for t in traits}
                        label = "1 bit, centroid" if b == 1 else f"{b} bits, {kind}"
                        results[label] = nearest_centroid({t: code[t][train_idx[t]] for t in traits},
                                                          {t: code[t][test_idx[t]] for t in traits}, traits, loo)
                    for label, by_trait in results.items():
                        for t, ok in by_trait.items():
                            BITS_ROWS += [{"run": run, "check": check, "code": label, "bits": budget, "seed": seed,
                                           "trait": t, "qid": qids[t][i], "correct": c}
                                          for i, c in zip(test_idx[t], ok)]


def boot_error(correct, rng):
    # Error rate and its 95% interval, resampling questions (each question's correctness averaged over seeds).
    x = 1 - np.asarray(correct, dtype=float)
    means = x[rng.integers(0, len(x), (BOOT, len(x)))].mean(1)
    return x.mean(), *np.percentile(means, [2.5, 97.5])


if BITS_ROWS:
    BR = pd.DataFrame(BITS_ROWS)
    per_q = BR.groupby(["check", "code", "bits", "run", "trait", "qid"], sort=False)["correct"].mean().reset_index()
    NORMS = pd.DataFrame(NORM_ROWS)
    per_q = per_q.merge(NORMS[["run", "trait", "qid", "stratum"]], on=["run", "trait", "qid"])
    rng = np.random.default_rng(0)
    CURVE = pd.DataFrame([{"check": c, "code": code, "bits": bits, "n": len(g),
                           **dict(zip(("error", "lo", "hi"), boot_error(g["correct"], rng)))}
                          for (c, code, bits), g in per_q.groupby(["check", "code", "bits"], sort=False)])
    fmt = lambda r: f"{r.error:.1%} ({r.lo:.1%}-{r.hi:.1%})"
    FL = pd.DataFrame(FLOOR_ROWS)
    FLOOR = {check: boot_error(g["correct"], rng) for check, g in FL.groupby("check", sort=False)}
    for check in CURVE["check"].unique():
        sub = CURVE[CURVE["check"] == check]
        print(f"{check}: misclassified share (95% interval), pooled over runs, n = {sub['n'].max()} questions")
        table = sub.assign(cell=sub.apply(fmt, axis=1)).pivot(index="bits", columns="code", values="cell")
        print(table.reindex(columns=[c for c in CODES if c in table]).fillna("-").to_string())
        e, lo, hi = FLOOR[check]
        wrong = int((~FL[FL["check"] == check]["correct"]).sum())
        one = sub[sub["code"] == "1 bit (RP-LSH)"].set_index("bits")["error"]
        print(f"exact cosine (the 1-bit curve's floor): {e:.1%} ({lo:.1%}-{hi:.1%}), {wrong} wrong")
        print("1 bit (RP-LSH) above that floor, in points:", {int(b): round(float(100 * (v - e)), 2) for b, v in one.items()})
        # If a question's bits are independent, the excess over the floor falls like bits^-0.5 once only
        # near-ties are left: the slope of log(excess) on log(bits) over the last sizes says how close that is.
        tail = one[one.index >= 256] - e
        tail = tail[tail > 0]
        if len(tail) >= 3:
            slope = np.polyfit(np.log2(tail.index.to_numpy(dtype=float)), np.log2(tail.to_numpy()), 1)[0]
            print(f"slope of log2(excess) per doubling from 256 bits: {slope:.2f} (independent bits and a flat spread of near-ties: -0.5)\n")
        else:
            print()

    # Per run: the smallest digest at which today's RP-LSH reaches 98% and 99%, and its error at 128 and 256 bits.
    rows = []
    for (check, run), g in per_q[per_q["code"] == "1 bit (RP-LSH)"].groupby(["check", "run"], sort=False):
        acc = g.groupby("bits")["correct"].mean()
        row = {"check": check, "run": run, "n": int(g.groupby("bits").size().max())}
        for target in (0.98, 0.99):
            row[f"bits for {target:.0%}"] = next((b for b in BUDGETS if acc.get(b, 0) >= target), "-")
        for bits in [b for b in (128, 256) if b in BUDGETS]:
            e, lo, hi = boot_error(g[g["bits"] == bits]["correct"], rng)
            row[f"error at {bits}"] = f"{e:.1%} ({lo:.1%}-{hi:.1%})"
        rows.append(row)
    print("Today's RP-LSH per run (1 bit, digest of the mean):")
    print(pd.DataFrame(rows).to_string(index=False))

    checks = list(CURVE["check"].unique())
    fig, axes = plt.subplots(len(checks), 2, figsize=(11, 3.6 * len(checks)), sharex=True, sharey=True, squeeze=False)
    styles = {"1 bit (RP-LSH)": (INK, "-", "o"), "1 bit, centroid": (MUTED, (0, (1, 1.5)), "."),
              **{f"{b} bits, {k}": (SLOTS[i], (0, (4, 2)) if k == "magnitude" else "-", "sDv^"[i])
                 for i, b in enumerate(DEPTHS) for k in ("angle", "magnitude")}}
    for row, check in enumerate(checks):
        for col, kind in enumerate(("angle", "magnitude")):
            ax = axes[row, col]
            for code in ["1 bit (RP-LSH)", "1 bit, centroid"] + [f"{b} bits, {kind}" for b in DEPTHS]:
                s = CURVE[(CURVE["check"] == check) & (CURVE["code"] == code)].sort_values("bits")
                color, ls, marker = styles[code]
                ax.plot(s["bits"], s["error"].clip(lower=1e-3), color=color, linestyle=ls, marker=marker,
                        markersize=4, label=code)
                ax.fill_between(s["bits"], s["lo"].clip(lower=1e-3), s["hi"].clip(lower=1e-3), color=color,
                                alpha=0.12, linewidth=0)
            ax.axhline(max(FLOOR[check][0], 1e-3), color=INK2, linewidth=1, linestyle=(0, (6, 2, 1, 2)), label="exact cosine")
            ax.set_xscale("log", base=2)
            ax.set_yscale("log")
            ax.set_xticks(BUDGETS, [f"{b}\n{b // 8} B" if b < 8192 else f"{b // 1024}k\n{b // 8192} KiB" for b in BUDGETS], fontsize=6)
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}" if v < 0.01 else f"{v:.0%}"))
            ax.set_title(f"{check}: b bits per projection, {kind}" + (" (keeps the norm)" if kind == "magnitude" else ""))
            ax.legend(fontsize=7, loc="lower left")
            if col == 0:
                ax.set_ylabel("misclassified (log)")
        axes[-1, 0].set_xlabel("digest size: bits, bytes")
        axes[-1, 1].set_xlabel("digest size: bits, bytes")
    fig.tight_layout()
    save(fig, "7d_bits")
    plt.show()

    # The same errors by magnitude: do weak (short) persona vectors miss more, and does a bigger digest help them?
    STRAT = pd.DataFrame([{"check": c, "code": code, "stratum": st, "bits": bits, "n": len(g),
                           **dict(zip(("error", "lo", "hi"), boot_error(g["correct"], rng)))}
                          for (c, code, st, bits), g in per_q[per_q["code"].isin(STRATA_CODES)]
                          .groupby(["check", "code", "stratum", "bits"], sort=False)])
    print("\nMagnitude strata: thirds of the vector norm within each run and trait (norm / that run and trait's median):")
    print(NORMS.groupby("stratum")["relative norm"].agg(["size", "min", "median", "max"]).reindex(STRATA).round(2).to_string())
    for check in checks:
        for code in STRATA_CODES:
            sub = STRAT[(STRAT["check"] == check) & (STRAT["code"] == code)]
            n = sub.groupby("stratum")["n"].max().reindex(STRATA)
            print(f"\n{check}, {code}: misclassified share (95% interval) by magnitude third, n = "
                  + " / ".join(f"{v:.0f}" for v in n))
            print(sub.assign(cell=sub.apply(fmt, axis=1)).pivot(index="bits", columns="stratum", values="cell")
                  .reindex(columns=STRATA).fillna("-").to_string())

    fig, axes = plt.subplots(len(checks), len(STRATA_CODES), figsize=(4.4 * len(STRATA_CODES), 3.4 * len(checks)),
                             sharex=True, sharey=True, squeeze=False)
    for row, check in enumerate(checks):
        for col, code in enumerate(STRATA_CODES):
            ax = axes[row, col]
            for i, st in enumerate(STRATA):
                sub = STRAT[(STRAT["check"] == check) & (STRAT["code"] == code) & (STRAT["stratum"] == st)].sort_values("bits")
                ax.plot(sub["bits"], sub["error"].clip(lower=1e-3), color=SLOTS[i], linestyle=["-", (0, (4, 2)), (0, (1, 1.5))][i],
                        marker="osD"[i], markersize=4, label=f"{st} third")
                ax.fill_between(sub["bits"], sub["lo"].clip(lower=1e-3), sub["hi"].clip(lower=1e-3), color=SLOTS[i],
                                alpha=0.12, linewidth=0)
            ax.set_xscale("log", base=2)
            ax.set_yscale("log")
            ax.set_xticks(BUDGETS, [f"{b}\n{b // 8} B" if b < 8192 else f"{b // 1024}k\n{b // 8192} KiB" for b in BUDGETS], fontsize=5)
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}" if v < 0.01 else f"{v:.0%}"))
            ax.set_title(f"{check}: {code}", fontsize=9)
            ax.legend(fontsize=7, loc="lower left", title="vector norm", title_fontsize=7)
            if col == 0:
                ax.set_ylabel("misclassified (log)")
            if row == len(checks) - 1:
                ax.set_xlabel("digest size: bits, bytes")
    fig.suptitle("Error by vector magnitude (thirds within run and trait)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save(fig, "7d_bits_by_magnitude")
    plt.show()

### 7e. Framings: which are stable, and which traits vary with them

Only for a drawn design (`selection.json`), where each question got one of its trait's 5 instruction pairs at
random. Three views, pooled over runs because each framing has only about 8 of a trait's 40 questions per run:

- **Elicitation by framing:** the share of a framing's questions that passed the gate. A framing that elicits
  far less than the others is a weak instruction, not a weak trait.
- **Classification by framing:** held-out and cross-validated accuracy of the questions asked with each framing.
- **Direction by framing:** the cosine between a framing's mean persona vector (its eliciting questions only)
  and the mean of the trait's *other* framings, at the default layer. Near 1 means the wording doesn't move the
  direction. A low or scattered value marks a framing, or a trait, whose direction depends on wording.

The spread across framings (standard deviation) ranks traits from stable to variable. Clusters and outliers within
a trait are future work (`PLAN.md`): this section only compares means.

In [ ]:
DRAWN = {run: d for run, d in DATA.items() if d["picks"]}
if not DRAWN:
    print("No drawn design here (no selection.json).")
else:
    fr = pd.DataFrame([{"run": run, "trait": t, "qid": q, "framing": p["framing"], "split": p["split"],
                        "eliciting": q in set(d["elic"].get(t, []))}
                       for run, d in DRAWN.items() for t in TRAITS for q, p in enumerate(d["picks"][t])])
    n_framings = int(fr["framing"].max()) + 1

    # Elicitation by framing, pooled over runs.
    rate = fr.groupby(["trait", "framing"])["eliciting"].mean().unstack().reindex(TRAITS)
    asked = fr.groupby(["trait", "framing"]).size().unstack().reindex(TRAITS)

    # Classification by framing, from the saved predictions (single layer).
    acc = None
    if PREDICTIONS:
        pr = pd.DataFrame([p | {"run": l, "check": "held-out" if k.endswith("held-out") else "cv"}
                           for l, by in PREDICTIONS["runs"].items() if l in DRAWN
                           for k, ps in by.items() if k.startswith("single") for p in ps if "framing" in p])
        if len(pr):
            pr["correct"] = pr["cosine_pred"] == pr["true"]
            acc = pr.groupby(["true", "framing"])["correct"].agg(["mean", "size"])

    # Direction by framing: cosine of each framing's mean with the mean of the trait's other framings.
    FRAMECOS = []
    if torch is not None:
        unit = lambda x: x / x.norm(dim=-1, keepdim=True)
        for run, d in DRAWN.items():
            for t in KEPT.get(run, []):
                path = OUT_DIR / run / f"all_{t}_diffs.pt"
                if not path.exists():
                    continue
                x = torch.load(path).float()
                layer = single_default(x.shape[1] - 1)
                elic = d["elic"].get(t, [])
                by_f = {f: [q for q in elic if d["picks"][t][q]["framing"] == f] for f in range(n_framings)}
                for f, qs in by_f.items():
                    rest = [q for q in elic if q not in qs]
                    if len(qs) >= 2 and len(rest) >= 2:
                        FRAMECOS.append({"run": run, "trait": t, "framing": f, "n": len(qs),
                                         "cos to other framings": float(unit(x[qs, layer].mean(0)) @ unit(x[rest, layer].mean(0)))})
    FC = pd.DataFrame(FRAMECOS)
    cosmap = (FC.groupby(["trait", "framing"])["cos to other framings"].mean().unstack().reindex(TRAITS)
              if len(FC) else None)

    panels = [("Eliciting share by framing", rate, SEQ, 0, 1, lambda v: f"{v:.0%}")]
    if cosmap is not None:
        lo = max(0.0, np.floor(np.nanmin(cosmap.to_numpy(dtype=float)) * 10) / 10 - 0.1)   # spread the colors
        panels.append((f"Cosine to the other framings' mean (colors from {lo:.1f})", cosmap, SEQ, lo, 1,
                       lambda v: f"{v:.2f}"))
    fig, axes = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 3.6), squeeze=False)
    for ax, (title, m, cmap, lo, hi, fmt) in zip(axes[0], panels):
        ax.imshow(m.to_numpy(dtype=float), cmap=cmap, vmin=lo, vmax=hi, aspect="auto")
        for a in range(m.shape[0]):
            for b in range(m.shape[1]):
                v = m.iat[a, b]
                if not pd.isna(v):
                    ax.text(b, a, fmt(v), ha="center", va="center", fontsize=8, color=ink_on(v - lo, hi - lo))
        ax.set_xticks(range(m.shape[1]), [f"pair {f}" for f in m.columns])
        ax.set_yticks(range(m.shape[0]), m.index)
        ax.grid(False)
        ax.set_title(title, fontsize=9)
    fig.suptitle(f"Framings, pooled over {len(DRAWN)} runs (instruction pairs 0-{n_framings - 1})", x=0.01, ha="left",
                 fontweight="bold")
    fig.tight_layout()
    save(fig, "7e_framings")
    plt.show()

    summary = pd.DataFrame({"eliciting share, mean": rate.mean(axis=1), "spread across framings (sd)": rate.std(axis=1),
                            "weakest framing": rate.idxmin(axis=1), "strongest framing": rate.idxmax(axis=1)})
    if cosmap is not None:
        summary["direction cos, mean"] = cosmap.mean(axis=1)
        summary["direction cos, min"] = cosmap.min(axis=1)
        summary["direction cos, sd over runs"] = FC.groupby("trait")["cos to other framings"].std().reindex(TRAITS)
    if acc is not None:
        summary["accuracy (cv + held-out)"] = pr.groupby("true")["correct"].mean().reindex(TRAITS)
    print("Per trait, from stable to variable (by the spread of the eliciting share):")
    print(summary.sort_values("spread across framings (sd)").to_string(float_format=lambda v: f"{v:.2f}"))
    print("\nQuestions asked per trait and framing (all runs):")
    print(asked.to_string())
    if acc is not None:
        print("\nAccuracy by trait and framing (single layer, cosine; cv and held-out together):")
        print(acc["mean"].unstack().reindex(TRAITS).to_string(float_format=lambda v: f"{v:.0%}"))
    if len(FC):
        print("\nLowest framing-to-rest cosines (a framing that moves the direction):")
        print(FC.nsmallest(8, "cos to other framings").round(3).to_string(index=False))

## 8. Pairs: do persona vectors carry over?

From `pairs.json`, on the traits both runs keep: the cosine between the two models' mean persona vectors per trait
(same layer), and transfer accuracy, where one model's personas classify the other's eliciting questions.

In [ ]:
valid = {k: v for k, v in (PAIR_RESULTS or {}).items() if "cos(persona A, persona B)" in v}
for key, v in (PAIR_RESULTS or {}).items():
    if key not in valid:
        print(f"{key}: {v}")
if valid:
    fig, ax = plt.subplots(figsize=(7, 3.2))
    markers = ["o", "s", "D"]
    for i, (key, v) in enumerate(valid.items()):
        cos = v["cos(persona A, persona B)"]
        ax.scatter([cos[t] for t in cos], [TRAITS.index(t) + (i - (len(valid) - 1) / 2) * 0.18 for t in cos],
                   color=SLOTS[i], marker=markers[i % 3], s=40, label=key, zorder=3)
    ax.set_yticks(range(len(TRAITS)), TRAITS)
    ax.invert_yaxis()
    ax.set_xlim(min(0, ax.get_xlim()[0]), 1)
    ax.axvline(0, color=BASELINE, linewidth=1)
    ax.set_xlabel("cosine between the two models' mean persona vectors")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="lower left", fontsize=8)
    ax.set_title("Persona direction kept within each pair (traits both keep)")
    save(fig, "8_pairs")
    plt.show()
    rows = []
    for key, v in valid.items():
        for direction, who in (("A personas -> B", f"{v['A']} -> {v['B']}"), ("B personas -> A", f"{v['B']} -> {v['A']}")):
            r = v[direction]
            rp = next(k for k in r if k.startswith("rp"))
            rows.append({"personas -> questions": who, "layer": v["layer"], "traits": len(v["traits"]),
                         "chance": v["chance"], "cosine": r["cosine"], rp: r[rp], "n": r["n"]})
    print(pd.DataFrame(rows).to_string(index=False, formatters={c: pct for c in ("chance", "cosine", rp)}))

## 9. The judges against each other

gpt-5.4 never scored these six rubrics, so there is no outside reference as in persona-lsh's section 9: the
evidence that a rubric works is that two judges of different families read it the same way. The gate in sections
2-8 is the one `hexaco-lsh.ipynb` section 4 wrote (`gate by` in `experiment.json`: gemma-qat). Here:

- **Eliciting question pairs per run, by judge**, next to the run's gate.
- **Agreement, per pair of judges:** the share of question pairs on the same side of the gate, Cohen's κ on the
  gate (agreement beyond chance), and Spearman ρ between their scores for the high-pole reply.
- **Per factor:** the same three numbers for each rubric, which is where a weak rubric would show.

Whether the judge changes the classification is section 9b.

In [ ]:
def judge_gate(s):
    return (isinstance(s["pos_score"], int) and isinstance(s["neg_score"], int)
            and s["pos_score"] >= POS_THRESHOLD and s["neg_score"] < NEG_THRESHOLD)


BY_JUDGE = {run: {j: read_json(OUT_DIR / run / "judges" / f"{j}.json") for j in JUDGES} for run in DATA}
BY_JUDGE = {run: {j: s for j, s in v.items() if s} for run, v in BY_JUDGE.items()}
judges_here = [j for j in JUDGES if any(j in v for v in BY_JUDGE.values())]
if len(judges_here) < 2:
    print("One judge for this experiment:", judges_here or JUDGES, "- nothing to compare.")
else:
    rows = []
    for run, per in BY_JUDGE.items():
        for j, scores in per.items():
            for t in TRAITS:
                for s in scores.get(t, []):
                    rows.append({"run": run, "trait": t, "qid": s["qid"], "judge": j, "gate": judge_gate(s),
                                 "pos": s["pos_score"] if isinstance(s["pos_score"], int) else np.nan,
                                 "refused": "REFUSAL" in (s["pos_score"], s["neg_score"]),
                                 "error": any(str(s[k]).startswith("ERROR") for k in ("pos_score", "neg_score"))})
    JR = pd.DataFrame(rows)
    counts = JR.groupby(["run", "judge"], sort=False)["gate"].sum().unstack().reindex(columns=judges_here)
    counts[f"the run's gate ({DESIGN.get('gate by', 'consensus')})"] = [sum(len(v) for v in DATA[r]["elic"].values()) for r in counts.index]
    print("Eliciting question pairs per run (of", JR.groupby(["run", "judge"]).size().max(), "):")
    print(counts.to_string())
    print("\nRefusals and errors per judge:", JR.groupby("judge", sort=False)[["refused", "error"]].sum().to_dict("index"))

    wide = JR.pivot_table(index=["run", "trait", "qid"], columns="judge", values=["gate", "pos"], aggfunc="first")
    agree = []
    for i, a in enumerate(judges_here):
        for b in judges_here[i + 1:]:
            ga, gb = wide["gate"][a].astype(bool), wide["gate"][b].astype(bool)
            po = (ga == gb).mean()
            pe = ga.mean() * gb.mean() + (1 - ga.mean()) * (1 - gb.mean())
            both = wide["pos"][[a, b]].dropna()
            agree.append({"judges": f"{a} / {b}", "same gate": f"{po:.0%}", "kappa": round((po - pe) / (1 - pe), 2),
                          "rho (persona score)": round(both[a].rank().corr(both[b].rank()), 2),
                          "mean score diff (a - b)": round((both[a] - both[b]).mean(), 1), "n": len(ga)})
    print("\nAgreement:")
    print(pd.DataFrame(agree).to_string(index=False))

    per_rubric = []
    for i, a in enumerate(judges_here):
        for b in judges_here[i + 1:]:
            for t in TRAITS:
                if t not in wide.index.get_level_values("trait"):
                    continue
                w = wide.xs(t, level="trait")
                ga, gb = w["gate"][a].astype(bool), w["gate"][b].astype(bool)
                po = (ga == gb).mean()
                pe = ga.mean() * gb.mean() + (1 - ga.mean()) * (1 - gb.mean())
                both = w["pos"][[a, b]].dropna()
                per_rubric.append({"judges": f"{a} / {b}", "factor": t, "same gate": f"{po:.0%}",
                                   "kappa": round((po - pe) / (1 - pe), 2) if pe < 1 else np.nan,
                                   "rho (high-pole score)": round(both[a].rank().corr(both[b].rank()), 2),
                                   f"passes {a}": int(ga.sum()), f"passes {b}": int(gb.sum()), "n": len(ga)})
    print("\nAgreement per rubric:")
    print(pd.DataFrame(per_rubric).to_string(index=False))

    per_trait = JR.groupby(["trait", "judge"], sort=False)["gate"].mean().unstack().reindex(index=TRAITS, columns=judges_here)
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for i, j in enumerate(judges_here):
        ax.scatter(per_trait[j], np.arange(len(TRAITS)) + (i - (len(judges_here) - 1) / 2) * 0.2, color=SLOTS[i],
                   marker="osD^v"[i % 5], s=36, label=j, zorder=3)
    ax.set_yticks(range(len(TRAITS)), TRAITS)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_xlabel("share of question pairs passing that judge's gate (all runs)")
    ax.grid(axis="y", visible=False)
    ax.legend(fontsize=8, loc="lower left")
    ax.set_title("Elicitation by trait, per judge")
    save(fig, "9_judges")
    plt.show()

    if torch is not None:
        res = []
        unit = lambda x: x / x.norm(dim=-1, keepdim=True)
        for run, per in BY_JUDGE.items():
            if not all((OUT_DIR / run / f"all_{t}_diffs.pt").exists() for t in TRAITS):
                continue
            diffs = {t: torch.load(OUT_DIR / run / f"all_{t}_diffs.pt").float() for t in TRAITS}
            layer = single_default(diffs[TRAITS[0]].shape[1] - 1)
            picks = DATA[run]["picks"]
            for j in ["consensus", *per]:
                scores = DATA[run]["scores"] if j == "consensus" else per[j]
                elic = {t: [s["qid"] for s in scores.get(t, []) if judge_gate(s)] for t in TRAITS}
                test_n = {t: sum(1 for q in elic[t] if not picks or picks[t][q]["split"] == "test") for t in TRAITS}
                traits = [t for t in TRAITS if test_n[t] >= MIN_ELICITING]
                if len(traits) < 2:
                    res.append({"run": run, "judge": j, "traits": len(traits)})
                    continue
                sums = {t: diffs[t][elic[t], layer].sum(0) for t in traits}
                ok = n = 0
                for t in traits:
                    for i, q in enumerate(elic[t]):
                        v = diffs[t][q, layer]
                        means = {u: (sums[u] - v) if u == t else sums[u] for u in traits}
                        ok += max(traits, key=lambda u: float(unit(means[u]) @ unit(v))) == t
                        n += 1
                res.append({"run": run, "judge": "the run's gate" if j == "consensus" else j, "traits": len(traits), "chance": f"{1 / len(traits):.0%}",
                            "n": n, "cosine, leave-one-out": f"{ok / n:.1%}"})
        print("\nClassification under each judge's gate (default layer):")
        print(pd.DataFrame(res).fillna("-").to_string(index=False))

### 9b. Does the choice of judge change the classification?

Needs the `.pt` vectors. Each gate picks its own eliciting questions, so each gets its own means and its own kept
factors (`MIN_ELICITING` eliciting test questions, as in section 3). At the default layer, with exact cosine:
leave-one-out over every eliciting question and held-out (means from the test split, the eval split classified).
Gates: each judge alone, the gate the run used, and every judge passing.

Accuracies across gates aren't on the same questions or always the same factors (chance is printed). The second
table is the like-for-like test. **Disputed questions**: means are built only from questions every judge
passes, and every other question of those factors is classified against them, grouped by which judges passed
it. If the questions only one judge passes classify like the agreed ones, that judge is right to let them in.
If they classify like the questions nobody passes (7b), its extra questions are noise.

In [ ]:
GATE_ROWS, DISPUTED = [], []
BOTH = [j for j in ("gemma-qat", "gpt-oss") if j in judges_here]
if torch is None:
    print("No torch in this kernel: section 9b needs torch.")
elif len(judges_here) < 2:
    print("One judge for this experiment: nothing to compare.")
else:
    unit = lambda x: x / x.norm(dim=-1, keepdim=True)
    for run in DATA:
        per = BY_JUDGE.get(run, {})
        if not all(j in per for j in judges_here) or not all((OUT_DIR / run / f"all_{t}_diffs.pt").exists() for t in TRAITS):
            continue
        diffs = {t: torch.load(OUT_DIR / run / f"all_{t}_diffs.pt").float() for t in TRAITS}
        layer = single_default(diffs[TRAITS[0]].shape[1] - 1)
        finite = {t: {q for q in range(diffs[t].shape[0]) if torch.isfinite(diffs[t][q, layer]).all()} for t in TRAITS}
        picks = DATA[run]["picks"]
        passed = {j: {t: {s["qid"] for s in per[j].get(t, []) if judge_gate(s)} & finite[t] for t in TRAITS} for j in judges_here}
        gates = dict(passed)
        gates[f"the run's gate ({DESIGN.get('gate by', 'consensus')})"] = {t: set(DATA[run]["elic"].get(t, [])) & finite[t] for t in TRAITS}
        if len(BOTH) == 2 and len(judges_here) > 2:
            gates[" and ".join(BOTH)] = {t: passed[BOTH[0]][t] & passed[BOTH[1]][t] for t in TRAITS}
        everyone = {t: set.intersection(*(passed[j][t] for j in judges_here)) for t in TRAITS}
        gates["every judge"] = everyone

        def in_test(t, q):
            return not picks or picks[t][q]["split"] == "test"

        for gate, sets in gates.items():
            traits = [t for t in TRAITS if sum(in_test(t, q) for q in sets[t]) >= MIN_ELICITING]
            row = {"run": run, "gate": gate, "eliciting": sum(len(v) for v in sets.values()), "traits": len(traits)}
            if len(traits) >= 2:
                qids = {t: sorted(sets[t]) for t in traits}
                X = {t: diffs[t][qids[t], layer] for t in traits}
                everything = {t: list(range(len(qids[t]))) for t in traits}
                ok = nearest_cosine(X, everything, everything, traits, True)
                row |= {"loo right": sum(map(sum, ok.values())), "loo n": sum(map(len, ok.values()))}
                if picks:
                    train = {t: [i for i, q in enumerate(qids[t]) if picks[t][q]["split"] == "test"] for t in traits}
                    test = {t: sorted(set(everything[t]) - set(train[t])) for t in traits}
                    ok = nearest_cosine(X, train, test, traits, False)
                    row |= {"held right": sum(map(sum, ok.values())), "held n": sum(map(len, ok.values()))}
            GATE_ROWS.append(row)

        # Disputed questions against the means of the questions every judge passes.
        traits = [t for t in TRAITS if len(everyone[t]) >= MIN_ELICITING]
        if len(traits) >= 2:
            M = torch.stack([unit(diffs[t][sorted(everyone[t]), layer].mean(0)) for t in traits])
            for k, t in enumerate(traits):
                for q in sorted(finite[t] - everyone[t]):
                    who = [j for j in judges_here if q in passed[j][t]]
                    group = ("no judge passes" if not who else f"only {who[0]} passes" if len(who) == 1
                             else "all but " + ", ".join(j for j in judges_here if j not in who) + " pass")
                    DISPUTED.append({"run": run, "trait": t, "qid": q, "group": group, "chance": 1 / len(traits),
                                     "correct": int((M @ unit(diffs[t][q, layer])).argmax()) == k})
            qids = {t: sorted(everyone[t]) for t in traits}
            idx = {t: list(range(len(qids[t]))) for t in traits}
            ok = nearest_cosine({t: diffs[t][qids[t], layer] for t in traits}, idx, idx, traits, True)
            DISPUTED += [{"run": run, "trait": t, "qid": q, "group": "every judge passes (leave-one-out)",
                          "chance": 1 / len(traits), "correct": c} for t in traits for q, c in zip(qids[t], ok[t])]

if GATE_ROWS:
    GR = pd.DataFrame(GATE_ROWS)
    share = lambda right, n: "-" if not n or pd.isna(n) else f"{right / n:.1%} ({int(right)}/{int(n)})"
    cells = GR.assign(**{"leave-one-out": [share(r, n) for r, n in zip(GR.get("loo right", np.nan), GR.get("loo n", np.nan))],
                         "held-out": [share(r, n) for r, n in zip(GR["held right"], GR["held n"])] if "held n" in GR else "-"})
    print("Per run and gate (eliciting pairs, traits kept, exact cosine at the default layer):")
    print(cells[["run", "gate", "eliciting", "traits", "leave-one-out", "held-out"]].to_string(index=False))
    pooled = GR.groupby("gate", sort=False).sum(numeric_only=True)
    pooled["leave-one-out"] = [share(r, n) for r, n in zip(pooled["loo right"], pooled["loo n"])]
    pooled["held-out"] = [share(r, n) for r, n in zip(pooled["held right"], pooled["held n"])] if "held n" in pooled else "-"
    print("\nPooled over runs:")
    print(pooled[["eliciting", "traits", "leave-one-out", "held-out"]].to_string())

if DISPUTED:
    DQ = pd.DataFrame(DISPUTED)
    by_group = DQ.groupby("group").agg(n=("correct", "size"), accuracy=("correct", "mean"), chance=("chance", "mean"))
    by_group = by_group.sort_values("accuracy", ascending=False)
    print("\nDisputed questions, classified against the means of the questions every judge passes:")
    print(by_group.to_string(formatters={"accuracy": pct, "chance": pct, "n": "{:.0f}".format}))
    print("\nBy run (correct / n):")
    print(DQ.groupby(["group", "run"])["correct"].agg(lambda s: f"{s.sum()}/{len(s)}").unstack().fillna("").to_string())

    fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(by_group) + 1.2))
    y = np.arange(len(by_group))[::-1]
    ax.barh(y, by_group["accuracy"], color=BLUE, height=0.55)
    ax.scatter(by_group["chance"], y, marker="|", s=220, color=MUTED, linewidths=2, zorder=3, label="chance (mean)")
    for yi, (acc, n) in zip(y, zip(by_group["accuracy"], by_group["n"])):
        ax.text(acc, yi, f"  {acc:.0%} of {n:.0f}", va="center", fontsize=8, color=INK2)
    ax.set_yticks(y, by_group.index)
    ax.set_xlim(0, 1.18)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right", fontsize=8)
    ax.set_title("Classified as the instructed trait, by which judges passed the question")
    save(fig, "9b_disputed")
    plt.show()

## 10. The cards during the run

From the run's telemetry directory: `events.csv` (each step's start and end) and `gpus.csv` (per card, once a
second). Time per run is split into capture (load, generation, hidden-state read) and judging. Power is the sum
over the model cards per sample. Kernel lines (`kernel.log`) are placed at the step that was running.

In [ ]:
def has_capture(d):
    # A rerun that only redid the analysis has no " gen" steps; its telemetry says nothing about capture.
    try:
        return pd.read_csv(d / "events.csv")["step"].astype(str).str.endswith(" gen").any()
    except (OSError, KeyError, pd.errors.ParserError):
        return False


tdir = (TELEMETRY_DIR if TELEMETRY_DIR and has_capture(TELEMETRY_DIR) else
        max((d for d in TELEMETRY_ROOT.glob("hexaco-lsh_*") if has_capture(d)), default=None, key=lambda p: p.name))
if not tdir or not (tdir / "events.csv").exists():
    print("No telemetry directory under", TELEMETRY_ROOT)
else:
    print("telemetry:", tdir)
    ev = pd.read_csv(tdir / "events.csv")
    cards = [int(c) for c in str(ev["card"].iloc[0]).split("+")]
    windows, starts = [], {}
    for e in ev.itertuples():
        if e.event == "start":
            starts[e.step] = e.epoch
        elif e.event in ("end", "error") and e.step in starts:
            windows.append({"step": e.step, "t0": starts.pop(e.step), "t1": e.epoch, "event": e.event})
    W = pd.DataFrame(windows)
    W["seconds"] = W["t1"] - W["t0"]

    def classify(step):
        for run in sorted(DATA, key=len, reverse=True):   # longest first: "qwen3-8b-think" before "qwen3-8b"
            if step.startswith(f"judge {run} "):
                return run, "judge"
            if step.startswith(f"{run} "):
                return run, "read" if step.endswith(" read") else "load" if step.endswith(" load") else "generate"
        return None, step.split()[0]

    W[["run", "phase"]] = W["step"].apply(lambda s: pd.Series(classify(s)))
    phases = ["load", "generate", "read", "judge"]
    hours = W.dropna(subset=["run"]).pivot_table(index="run", columns="phase", values="seconds", aggfunc="sum",
                                                 fill_value=0).reindex(columns=phases, fill_value=0) / 3600
    hours = hours.reindex([r for r in DATA if r in hours.index])

    g = pd.read_csv(tdir / "gpus.csv", skipinitialspace=True, on_bad_lines="skip")
    g = g[pd.to_numeric(g["index"], errors="coerce").notna()].astype({"index": int})
    g["power.draw"] = pd.to_numeric(g["power.draw"], errors="coerce")
    g["temperature.gpu"] = pd.to_numeric(g["temperature.gpu"], errors="coerce")
    mine = g[g["index"].isin(cards)]
    total = mine.groupby("epoch")["power.draw"].agg(["sum", "count"])
    total = total[total["count"] == len(cards)]["sum"]

    def in_window(series, t0, t1):
        return series[(series.index >= t0) & (series.index <= t1)]

    rows = []
    for run in hours.index:
        w = W[W["run"] == run]
        cap = pd.concat([in_window(total, r.t0, r.t1) for r in w[w["phase"] != "judge"].itertuples()])
        temps = mine[mine["epoch"].between(w["t0"].min(), w[w["phase"] != "judge"]["t1"].max())]
        rows.append({"run": run, **{f"{p} h": hours.loc[run, p] for p in phases},
                     "capture mean W": cap.mean(), "capture max W": cap.max(),
                     **{f"card {c} max C": temps[temps["index"] == c]["temperature.gpu"].max() for c in cards}})
    TEL = pd.DataFrame(rows)
    print(TEL.round(2).to_string(index=False))
    print(f"\nwhole run: {(ev['epoch'].max() - ev['epoch'].min()) / 3600:.1f} h; "
          f"total W across cards {'+'.join(map(str, cards))}: mean {total.mean():.0f}, max {total.max():.0f}")

    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), gridspec_kw={"height_ratios": [1, 1.1]})
    ax = axes[0]
    y = np.arange(len(hours))[::-1]
    left = np.zeros(len(hours))
    for i, p in enumerate(phases):
        ax.barh(y, hours[p], left=left, color=SLOTS[i], height=0.6, edgecolor=SURFACE, linewidth=1, label=p)
        left += hours[p].to_numpy()
    for yi, h in zip(y, left):
        ax.text(h, yi, f"  {h:.1f} h", va="center", fontsize=8, color=INK2)
    ax.set_yticks(y, hours.index)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("hours")
    ax.set_xlim(0, left.max() * 1.15)
    ax.legend(loc="lower right", ncol=4)
    ax.set_title("Wall time per run")

    ax = axes[1]
    start = ev["epoch"].min()
    smooth = total.rolling(60, min_periods=1).mean()
    ax.plot((total.index - start) / 3600, total.to_numpy(), color=BLUE, linewidth=0.4, alpha=0.35)
    ax.plot((smooth.index - start) / 3600, smooth.to_numpy(), color=BLUE, linewidth=1.5,
            label=f"cards {'+'.join(map(str, cards))}, 60 s mean")
    spans = W.dropna(subset=["run"]).groupby("run").agg(t0=("t0", "min"), t1=("t1", "max"))
    for i, run in enumerate(r for r in DATA if r in spans.index):
        if spans.loc[run, "t1"] - spans.loc[run, "t0"] > 0:
            capture = W[(W["run"] == run) & (W["phase"] != "judge")]
            if len(capture):
                a, b = (capture["t0"].min() - start) / 3600, (capture["t1"].max() - start) / 3600
                ax.axvspan(a, b, color=GRID, alpha=0.6 if i % 2 else 0.25, linewidth=0)
                ax.text((a + b) / 2, 1.0, run, transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                        fontsize=7, color=INK2, rotation=30)
    log = [l for l in (tdir / "kernel.log").read_text().splitlines() if l.strip()]
    KERNEL = []
    for line in log:
        try:
            t = float(line.split()[0])
        except ValueError:
            continue
        hit = W[(W["t0"] <= t) & (W["t1"] >= t)]
        KERNEL.append({"hours in": (t - start) / 3600, "step": "; ".join(hit["step"]) or "(between steps)",
                       "line": " ".join(line.split()[2:])[:110]})
    for i, t in enumerate(sorted({round(k["hours in"], 3) for k in KERNEL})):
        ax.axvline(t, color=SERIOUS, linewidth=1, linestyle=(0, (2, 2)),
                   label="kernel message (AER/NVRM)" if i == 0 else None)
    ax.set_xlabel("hours since the notebook started")
    ax.set_ylabel("watts")
    ax.legend(loc="lower left", fontsize=8)
    ax.set_title("Total power on the model cards (shaded: each run's capture)", pad=34)
    fig.tight_layout()
    save(fig, "10_telemetry")
    plt.show()

    print("\nHighest one-second samples per card, and the step running:")
    for c in cards:
        top = mine[mine["index"] == c].nlargest(3, "power.draw")
        for _, r in top.iterrows():
            hit = W[(W["t0"] <= r["epoch"]) & (W["t1"] >= r["epoch"])]
            print(f"  card {c}: {r['power.draw']:.0f} W (limit {r['power.limit']}) at "
                  f"{(r['epoch'] - start) / 3600:.2f} h, {'; '.join(hit['step']) or '(between steps)'}")

    if KERNEL:
        print(f"{len(KERNEL)} kernel lines:")
        print(pd.DataFrame(KERNEL).round(2).to_string(index=False))
        buses = g.drop_duplicates("index").set_index("pci.bus_id")["index"].to_dict()
        print("bus -> card:", {b[-7:]: c for b, c in buses.items()})
    else:
        print("kernel messages: none")

## 11. Interpretation

### 11a. `rand40-t256-s68853742` (2026-11-24): what `hexaco-lsh.ipynb` already shows

These numbers are from the saved outputs of `hexaco-lsh.ipynb` (its sections 4-7), not from this notebook, which
hasn't run on the experiment yet. Sections 2, 2b, 4b, 5, 5b, 7, 9 and 10 need the output directory; 11b lists
what to read off them.

**The run.** Llama-3.1-8B, Granite-4.2-8B, Bolt-E4B, Nemotron-Nano-8B and Qwen3-8B (thinking off), in that
order, on cards 0-2 in fp16. Capture took about an hour a model (08:30 to about 14:15), the two judges about
2.4 hours after it, and the notebook finished at 16:52: 8.5 hours.

**Both poles are elicited, in every model and factor.** gemma-qat's gate passes 991 of 1,200 question pairs
(83%), and every run keeps all six factors, so chance is 1/6 throughout.
- Per run, of 240: Bolt 232, Llama 221, Qwen3 215, Granite 190, Nemotron-Nano 133.
- Per factor, of 200 over the five runs: extraversion 187, neuroticism 184, openness 165, agreeableness 159,
  conscientiousness 153, honesty-humility 143.
- The thin cells, of 40: Nemotron-Nano conscientiousness 9 (4 test, 5 eval: at `MIN_ELICITING`, so its mean
  rests on 4 questions), Granite honesty-humility 17, Nemotron-Nano honesty-humility 18 and agreeableness 22.
  Nemotron-Nano is the bland model again (`persona-hexaco`: everything about 3.2 of 5).
- The two factors where *with* is the departure or is neutral (N, E) elicit best. The three where *without* is
  the instructed departure toward an unwelcome pole (C, A, H) elicit worst. Section 2 says which reply fails.

**The two judges agree on the totals.** Eliciting pairs per run, gemma-qat / gpt-oss: Llama 221 / 227, Granite
190 / 184, Bolt 232 / 229, Nemotron-Nano 133 / 130, Qwen3 215 / 217; 991 against 987 overall, with 3 judge
errors (all gpt-oss). Totals can match while the questions differ: section 9 has the per-question agreement.

**The six factors are separable, less cleanly than persona-lsh's seven traits.** Single default layer, chance
17%:

| | cosine | RP-LSH 1024 | RP-LSH 256 | n |
|---|---|---|---|---|
| 3-fold cross-validation | 94.6% | 94.0% | 91.5% | 496 |
| held-out eval set | 96.0% | 95.6% | 92.9% | 495 |

- Per run, cosine, cross-validated / held out: Granite 98% / 100%, Qwen3 97% / 98%, Bolt 94% / 94%, Llama 94% /
  94%, Nemotron-Nano 88% / 94%.
- persona-lsh on the same five models with the same procedure (`rand40-t256-s52466129`, its section 11a): 98.1%
  and 99.3%. So the factors cost about 3.5 points, which is roughly three times the errors (27 of 496 against 8
  of 421 in cross-validation). Section 4b puts the runs side by side.
- Against the self-report run: in `persona-hexaco`, nearest-mean identified which factor an inventory item was
  about 20-63% of the time, and the factor directions leaked into each other at 0.5-1.0. The tasks differ (an
  item's factor there, the instructed factor of an open reply here), but the contrast is large: with instructed
  replies the same models separate the six factors 94-96% of the time. Whether the six directions still share
  an axis is section 7a's question.

**RP-LSH keeps it at 1024 bits and loses about 3 points at 256.** At 1024 bits the digest is within 0.6 points
of cosine on both checks. At 256 bits it is 3.1 points under. Section 7d gives the curve, its floor, and the
size at which each run reaches 98%.

**The layer strategies don't matter.** `single`, `multi` and `all` are within about 2 points of each other in
every run, with no consistent order.

**Depth: the signal is there early in three models, and persona-lsh's default layer is near the best.**
- Best single layer: Llama 19 of 32, Granite 27 of 40, Bolt 29 of 42, all within a few layers of the default
  (20/28 of depth); Nemotron-Nano and Qwen3 peak at or next to the last layer, by a point.
- Only Granite (layers 25-39) and Qwen3 (20-36) reach 95% at any layer.
- Layer 1 already classifies at 94% in Granite, 83% in Qwen3 and 79% in Bolt, against 69% in Llama and 54% in
  Nemotron-Nano. Granite then dips to 87% at layer 8 before rising. The first layers are close to the reply's
  tokens, so a high start probably means the two poles already differ in wording in those models. Not tested.

**Transfer within the Llama pair holds.** Llama-3.1-8B and Nemotron-Nano-8B at layer 23: the two models' factor
directions have cosine 0.63 (agreeableness) to 0.77 (extraversion). Llama's means classify Nemotron-Nano's
eliciting questions at 91.0% (RP-LSH 1024: 90.2%, n = 133), and the reverse at 93.2% (92.3%, n = 221). That is
as good as each model's own cross-validation, so a factor library built on the base model reads its fine-tune.
The other three pairs weren't in this pass.

### 11b. To read off once this notebook has run on the experiment

- **Which pole fails (2).** Per factor, `with` below 50 against `without` 50+ and refusals. The prediction: C, A
  and H fail on the low pole (the model stays conscientious, kind or honest when told not to), and refusals
  cluster at low A and low H. Granite's honesty-humility (17 of 40) is the cell to look at first.
- **Which questions to rewrite (2b).** The questions failing in three or more runs, and whether they share a
  facet or an instruction pair. This settles the open item in `PLAN.md` section 9.
- **Which factors are confused (5, 5b).** Is it agreeableness with honesty-humility, as the literature
  predicts, or everything with neuroticism, as in the self-report run?
- **Six directions or one axis (7, 7a).** The three predicted signs, the desirable-keyed cosines and the first
  component's share. A share near 17% with small cosines says six directions; a large share with neuroticism
  loading the other way says a shared axis under the six, with the 94-96% accuracy coming from what is left
  around it.
- **Instruction or behavior (7b), and magnitude (7c).** Do the 209 non-eliciting question pairs still classify
  as their instructed factor, and does the projection grade the judge's score?
- **Digest size (7d).** The bits at which each run reaches 98%, against persona-lsh's.
- **Instruction pairs (7e).** Each pair of a factor is written on different facets, so a pair whose direction
  differs from the others' may be a facet, not a wording.
- **Judges per rubric (9, 9b).** κ and ρ per factor. These rubrics have no gpt-5.4 check, so a rubric where the
  two judges part is one to rewrite before it is used further.
- **Then** `trait-crossref.ipynb`, rebuilt for these directions, against persona-lsh's seven traits from the
  same models.